# GENZONE | AI Manthan - N-CMAPSS physics-informed EKF + stacking ensemble (training, per `pipeline.md`)
Trains the pipeline specified in `pipeline.md` from the raw NASA N-CMAPSS files and exports one **model bundle**:

```text
N-CMAPSS HDF5 -> schema inspection + validation -> flight-end aggregation -> engine split (train / validation / test)
  -> healthy-condition baseline g(W) (cross-fitted residuals) -> reduced-order gas-path model + EKF (states, covariance,
  innovations) -> causal features -> Random Forest, XGBoost, LightGBM, CatBoost -> out-of-fold predictions
  -> scaler + Ridge -> RUL + empirical interval -> evaluation, ablation E0-E7, SHAP + Ridge contributions -> bundle
  (+ section 14, an EXTENSION beyond pipeline.md: component-health forecasts h flights ahead)
```

**Run on Google Colab (full guide: `training/README.md`)**
1. A CPU runtime is enough (no GPU is used). High-RAM helps for `RUN = "full"`.
2. Edit the configuration cell: `RUN = "quick"` (DS02 only, check the pipeline) or `RUN = "full"` (9 files).
3. Run all cells and accept the Google Drive prompt (cache of aggregated data, so re-runs skip the download).
4. Data comes from Kaggle (`bishals098/nasa-cmapss-2-engine-degradation`, public); add Colab secrets
   `KAGGLE_USERNAME` / `KAGGLE_KEY` only if the anonymous download fails.
5. The last cell downloads `genzone_ncmapss_<RUN>_bundle.zip` (a copy goes to Drive). Unzip it into `backend/models/`
   and run `python -m backend.check_parity`.

**Rules kept (pipeline.md section 0):** engines are split before anything is fitted; the official test engines are used
once, for the final evaluation; hidden degradation parameters `T` are used only offline (physics calibration, labels for
validating the health mapping), never as inputs; `Y` is only the target; all features are causal; nothing is fitted
on synthetic data; no guessed EKF values - everything is estimated from training data and saved in the bundle.

## 0. Setup

In [27]:
import gc
import glob
import hashlib
import json
import os
import platform
import shutil
import sys
import time
import warnings
import zlib

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/input")
T_START = time.time()
TIMINGS = {}


def stage(name):
    print(f"\n=== {name}   [{(time.time() - T_START) / 60:.1f} min elapsed]")


if IN_COLAB or IN_KAGGLE:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "xgboost", "lightgbm", "catboost", "shap", "h5py",
                    "kagglehub", "pyarrow"], check=False)
    try:    # make Colab secrets visible to kagglehub (only needed if the anonymous download fails)
        from google.colab import userdata
        for k in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
            try:
                os.environ[k] = userdata.get(k)
            except Exception:
                pass
    except Exception:
        pass

## 1. Configuration (edit this cell)

In [28]:
RUN = os.environ.get("RUN", "full")                          # "quick" (DS02) or "full" (9 files; DS08d excluded)
DATASET_HANDLE = "bishals098/nasa-cmapss-2-engine-degradation"
FILES = {"quick": ["N-CMAPSS_DS02-006.h5"],
         "full": ["N-CMAPSS_DS01-005.h5", "N-CMAPSS_DS02-006.h5", "N-CMAPSS_DS03-012.h5", "N-CMAPSS_DS04.h5",
                  "N-CMAPSS_DS05.h5", "N-CMAPSS_DS06.h5", "N-CMAPSS_DS07.h5", "N-CMAPSS_DS08a-009.h5",
                  "N-CMAPSS_DS08c-008.h5"]}[RUN]
# DS08d-010 is excluded: Kaggle's copy is truncated inside HDF5's internal index (cannot be read even after padding).
DATA_DIR = os.environ.get("NCMAPSS_DIR")                      # optional folder that already holds the .h5 files
DELETE_AFTER_USE = True                                       # delete each downloaded .h5 after aggregation (disk)
USE_DRIVE = True                                              # Colab: mount Drive for the aggregate cache
DRIVE_DIR = "/content/drive/MyDrive/genzone"                  # cache and a copy of the bundle go here
CACHE_AGGREGATES = True                                       # reuse per-file aggregates from earlier runs
HEALTHY_SAMPLES_PER_FILE = 200_000                            # 1 Hz healthy rows per file for the healthy baseline
SEED = 42
VAL_FRACTION = 0.2                                            # share of development engines held out for validation
N_FOLDS = 5                                                   # engine-grouped folds for out-of-fold predictions
ROLL = 5                                                      # trailing window (flights) for temporal features
MERGE_COS = 0.95                                              # EKF: merge states whose sensor signatures are this similar
MERGE_UNIQUE = 0.15                                           # EKF: merge a state whose signature is mostly the others'
EKF_GATE = 9.0                                                # EKF: per-sensor NIS gate (reject up to 3 flights in a row)
Q_GRID = (0.1, 0.3, 1.0, 3.0, 10.0)                           # EKF: process-noise scales tried on validation engines
TUNE = True                                                   # small validation-engine grids for the four models
INTERVAL_LEVEL = 0.90                                         # empirical RUL interval from validation residuals
RELIABLE_MAX_RMSE = 0.10                                      # component health shown as reliable if validation RMSE on
RELIABLE_MIN_CYCLES = 30                                      #   degraded flights (HI < 0.9) <= 10 points (>= 30 flights)
RUN_ABLATION = True                                           # E0-E7 (pipeline.md section 10.2)
RUN_FORECAST = True                                           # EXTENSION: component-health forecasts (section 14)
FORECAST_HORIZONS = (10, 25, 50)                              # flights ahead
FORECAST_DEGRADED_BELOW = 0.9                                 # score forecasts on flights whose current true health < this
SHAP_ROWS = 300                                               # training rows explained per model (Random Forest: <= 100)
BAND_EDGES = [0.75, 0.55, 0.30]                               # Normal / Warning / Alarm / Emergency health bands
BUILD_INFO = {'commit': '324fe10', 'code_sha256': '7530b96e9236db0f8dc7d5f84007775faf337b7520a36fb3baed0767dd6ea493'}            # filled in by training/build_notebook.py

## 2. Google Drive (aggregate cache)

In [29]:
DRIVE_OK = False
if USE_DRIVE and IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        os.makedirs(DRIVE_DIR, exist_ok=True)
        DRIVE_OK = True
    except Exception as exc:
        print("Google Drive not mounted -> no cache this session:", repr(exc))
CACHE_DIR = os.environ.get("NCMAPSS_CACHE") or (DRIVE_DIR if DRIVE_OK else None)
if CACHE_DIR:
    os.makedirs(CACHE_DIR, exist_ok=True)
print("cache folder:", CACHE_DIR or "none (nothing is reused between sessions)")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
cache folder: /content/drive/MyDrive/genzone


## 3. Shared pipeline code (identical to backend/pipeline/ncmapss.py and stacking.py - do not edit here)

In [30]:
"""NASA N-CMAPSS physics-informed pipeline, implemented as specified in pipeline.md.

Layers (all causal per engine, one prediction per completed flight = flight-end cadence):
  1  ingestion          inspect_h5, load_h5 (columns mapped BY NAME), validate_split, aggregate_cycles
  2  physics            physics_1hz (pressure ratios, efficiency proxies) + HealthyReference g(W) (healthy-condition
                        baseline) -> residual = measured / healthy - 1
  3  state estimation   GasPathEKF: reduced-order observation model  res = b(W) + B(W) x + v,  random-walk state x,
                        Kalman update with Joseph covariance, innovation gating; states that the sensors cannot
                        separate are merged (observability analysis)
  4  features           CausalFeaturizer: current values + trailing-window mean/std/min/max/slope, exposure

This ONE module is used by the Colab training notebook (inlined) and by the backend, so training and serving compute
identical features. Dependencies: numpy, pandas (+ h5py for reading .h5 files).

Conventions: hidden degradation parameters `T` are used ONLY offline (labels, physics calibration, validation), never
as online inputs; `Y` (RUL) is only a target. Health index 1.0 = healthy, 0.0 = failure-level damage of that component.
"""
from __future__ import annotations

import itertools
import os

import numpy as np
import pandas as pd

AGG_VERSION = 2          # bump when load_h5 / aggregate_cycles change: invalidates cached aggregates

# ----------------------------------------------------------------------------------------------- schema
W_VARS = ["alt", "Mach", "TRA", "T2"]
XS_VARS = ["T24", "T30", "T48", "T50", "P15", "P2", "P21", "P24", "Ps30", "P40", "P50", "Nf", "Nc", "Wf"]
XV_DISPLAY = ["T40", "P30", "P45"]   # virtual (unmeasured) sensors: kept for display only, never model inputs
T_VARS = ["fan_eff_mod", "fan_flow_mod", "LPC_eff_mod", "LPC_flow_mod", "HPC_eff_mod", "HPC_flow_mod",
          "HPT_eff_mod", "HPT_flow_mod", "LPT_eff_mod", "LPT_flow_mod"]
COMPONENTS = ["fan", "lpc", "hpc", "hpt", "lpt"]
COMP_THETA = {"fan": ("fan_eff_mod", "fan_flow_mod"), "lpc": ("LPC_eff_mod", "LPC_flow_mod"),
              "hpc": ("HPC_eff_mod", "HPC_flow_mod"), "hpt": ("HPT_eff_mod", "HPT_flow_mod"),
              "lpt": ("LPT_eff_mod", "LPT_flow_mod")}
COMP_NAMES = {"fan": "Fan", "lpc": "LPC", "hpc": "HPC", "hpt": "HPT", "lpt": "LPT"}

SENSOR_INFO = {   # description, kind ("T" degR, "P" psia, "N" rpm, "W" lbm/s)
    "T24": ("LPC outlet total temperature", "T"), "T30": ("HPC outlet total temperature", "T"),
    "T48": ("HPT outlet total temperature (EGT)", "T"), "T50": ("LPT outlet total temperature", "T"),
    "P15": ("Bypass duct total pressure", "P"), "P2": ("Fan inlet total pressure", "P"),
    "P21": ("Fan outlet total pressure", "P"), "P24": ("LPC outlet total pressure", "P"),
    "Ps30": ("HPC outlet static pressure", "P"), "P40": ("Burner outlet total pressure", "P"),
    "P50": ("LPT outlet total pressure", "P"), "Nf": ("Fan (LP spool) speed", "N"),
    "Nc": ("Core (HP spool) speed", "N"), "Wf": ("Fuel flow", "W"),
    "T40": ("Burner outlet temperature (virtual)", "T"), "P30": ("HPC outlet total pressure (virtual)", "P"),
    "P45": ("HPT outlet total pressure (virtual)", "P"),
}
UNITS = {"alt": "ft", "Mach": "-", "TRA": "%", "T2": "degR", "T": "degR", "P": "psia", "N": "rpm", "W": "lbm/s"}

GAMMA = 1.4
K_COLD = (GAMMA - 1.0) / GAMMA
GAMMA_HOT = 1.33
K_HOT = (GAMMA_HOT - 1.0) / GAMMA_HOT

# Physics proxies computed at 1 Hz from MEASURED signals only, and the sensors each one depends on (occlusion explainer).
PROXY_DEPS = {
    "fan_pr": ["P21", "P2"], "lpc_pr": ["P24", "P21"], "hpc_pr": ["Ps30", "P24"],
    "eta_core": ["P24", "P2", "T24"], "eta_hpc": ["Ps30", "P24", "T30", "T24"],
    "burner_dp": ["P40", "Ps30"], "lpt_tr": ["T50", "T48"], "turb_pr": ["P40", "P50"],
    "nf_corr": ["Nf"], "nc_corr": ["Nc", "T24"], "wf_corr": ["Wf", "P2"], "spool_ratio": ["Nc", "Nf"],
}
PROXIES = list(PROXY_DEPS)

# Fixed (not data-fitted) centring/scaling of the flight condition, so per-cycle polynomial means can be accumulated
# before any model is fitted.
W_CENTER = {"alt": (20000.0, 15000.0), "Mach": (0.5, 0.25), "TRA": (60.0, 20.0), "T2": (500.0, 30.0)}

DEG_R_TO_K = 5.0 / 9.0
PSIA_TO_KPA = 6.894757
FT_TO_M = 0.3048
LBM_TO_KG = 0.45359237


# ----------------------------------------------------------------------------------------------- Layer 1: ingestion
def h5_names(f, key: str) -> list[str]:
    return [str(s).strip() for s in np.array(f[key]).reshape(-1).astype("U32")]


def inspect_h5(path: str) -> dict:
    """Keys, shapes, dtypes and attributes of an HDF5 file (pipeline.md section 2.5, step 1)."""
    import h5py

    info = {"file": os.path.basename(path), "bytes": int(os.path.getsize(path)), "datasets": {}, "groups": [],
            "attrs": {}}
    with h5py.File(path, "r") as f:
        info["attrs"] = {str(k): str(v)[:200] for k, v in f.attrs.items()}

        def visit(name, obj):
            if hasattr(obj, "shape"):
                info["datasets"][name] = {"shape": [int(d) for d in obj.shape], "dtype": str(obj.dtype)}
            else:
                info["groups"].append(name)
        f.visititems(visit)
    return info


def load_h5(path: str, split: str = "dev", virtual=XV_DISPLAY) -> pd.DataFrame:
    """Read one split ("dev" or "test") of an N-CMAPSS .h5 file into a 1 Hz DataFrame (float32).

    Columns are mapped by the variable names stored in the file (W_var, X_s_var, ...), never by position; the groups
    must be row-aligned. Corrupt rows (non-finite W/X_s, sensor <= 0, unit/cycle < 1) are removed and counted in
    df.attrs["dropped_rows"]; nothing else is altered.
    """
    import h5py

    with h5py.File(path, "r") as f:
        frames, lengths = [], {}
        for grp in ("W", "X_s", "T", "A"):
            names = h5_names(f, f"{grp}_var")
            arr = np.asarray(f[f"{grp}_{split}"], dtype=np.int32 if grp == "A" else np.float32)
            if arr.ndim != 2 or arr.shape[1] != len(names):
                raise ValueError(f"{path} [{split}]: {grp} has shape {arr.shape} but {len(names)} variable names")
            lengths[grp] = arr.shape[0]
            frames.append(pd.DataFrame(arr, columns=names))
        y = np.asarray(f[f"Y_{split}"], dtype=np.float32).reshape(-1)
        lengths["Y"] = len(y)
        frames.append(pd.DataFrame({"RUL": y}))
        if virtual and f"X_v_{split}" in f:
            vnames = h5_names(f, "X_v_var")
            keep = sorted(vnames.index(v) for v in virtual if v in vnames)
            if keep:
                arr = np.asarray(f[f"X_v_{split}"][:, keep], dtype=np.float32)
                lengths["X_v"] = arr.shape[0]
                frames.append(pd.DataFrame(arr, columns=["v_" + vnames[i] for i in keep]))
    if len(set(lengths.values())) != 1:
        raise ValueError(f"{path} [{split}]: groups are not row-aligned: {lengths}")
    df = pd.concat(frames, axis=1)
    missing = [c for c in W_VARS + XS_VARS + ["unit", "cycle"] if c not in df.columns]
    if missing:
        raise ValueError(f"{path} [{split}] is missing variables {missing}")
    ws = df[W_VARS + XS_VARS].to_numpy()
    bad = ~np.isfinite(ws).all(axis=1) | (df[XS_VARS].to_numpy() <= 0).any(axis=1) | \
        (df["unit"].to_numpy() < 1) | (df["cycle"].to_numpy() < 1)
    if bad.any():
        df = df.loc[~bad].reset_index(drop=True)
    df.attrs["dropped_rows"] = int(bad.sum())
    return df


def validate_split(df: pd.DataFrame, split: str = "dev") -> dict:
    """Schema/ordering/label checks of a loaded split (pipeline.md section 2.5, steps 3-7).

    Raises ValueError for violations the pipeline depends on (engine rows not contiguous, cycles going backwards);
    everything else is reported (e.g. RUL not monotone, missing values) so it can be inspected in the bundle.
    """
    u, c = df["unit"].to_numpy(), df["cycle"].to_numpy()
    same = u[1:] == u[:-1]
    contiguous = len(np.unique(u)) == 1 + int(np.count_nonzero(~same))
    monotone = bool(np.all(c[1:][same] >= c[:-1][same]))
    if not contiguous:
        raise ValueError(f"[{split}] rows of an engine are not contiguous")
    if not monotone:
        raise ValueError(f"[{split}] cycle numbers go backwards within an engine")
    rep = {"split": split, "rows_1hz": int(len(df)), "engines": int(len(np.unique(u))),
           "dropped_rows": int(df.attrs.get("dropped_rows", 0)), "units": sorted(int(v) for v in np.unique(u))}
    if "RUL" in df:
        y = df["RUL"].to_numpy()
        rep["rul_non_increasing_within_engine"] = not bool(np.any(same & (y[1:] > y[:-1] + 1e-6)))
        rep["rul_min"], rep["rul_max"] = float(np.nanmin(y)), float(np.nanmax(y))
    if "hs" in df:
        rep["hs_values"] = sorted(int(v) for v in np.unique(df["hs"]))
    na = df.isna().sum()
    rep["missing_values"] = {k: int(v) for k, v in na.items() if v}
    rep["sensor_range"] = {s: [float(df[s].min()), float(df[s].max())] for s in XS_VARS}
    rep["condition_range"] = {w: [float(df[w].min()), float(df[w].max())] for w in W_VARS}
    return rep


def repair_truncated_h5(path: str, workdir: str, max_missing: int = 4096):
    """Some mirrors ship an .h5 a few bytes shorter than its header records (Kaggle's DS08d-010: 32 bytes), which HDF5
    refuses to open. Copy it to `workdir` and zero-pad the copy up to the recorded size (load_h5 then drops any zeroed
    rows). Returns (path to open, repaired copy to delete afterwards or None)."""
    import re
    import shutil

    import h5py

    try:
        h5py.File(path, "r").close()
        return path, None
    except OSError as exc:
        m = re.search(r"eof = (\d+), sblock->base_addr = \d+, stored_eof = (\d+)", str(exc))
        if not m or not 0 < int(m.group(2)) - int(m.group(1)) <= max_missing:
            raise
        missing = int(m.group(2)) - int(m.group(1))
    fixed = os.path.join(workdir, "repaired_" + os.path.basename(path))
    shutil.copyfile(path, fixed)
    with open(fixed, "ab") as fh:
        fh.write(b"\0" * missing)
    return fixed, fixed


def physics_1hz(df: pd.DataFrame) -> pd.DataFrame:
    """First-principles ratios from measured signals (unit-free; temperatures in degR, pressures in psia).

    Efficiencies are isentropic PROXIES (pipeline.md 4.3): eta_core spans fan-hub + LPC (T2,P2 -> T24,P24); eta_hpc uses
    the measured static Ps30 because total P30 is not measured. No station is invented.
    """
    def ratio(a, b):
        b = np.asarray(b, dtype=np.float64)
        return np.asarray(a, dtype=np.float64) / np.where(np.abs(b) < 1e-9, np.nan, b)

    def eta(pr, tr):
        pr = np.clip(pr, 1e-6, None)
        return np.clip(ratio(pr ** K_COLD - 1.0, tr - 1.0), 0.0, 1.5)

    out = pd.DataFrame(index=df.index)
    out["fan_pr"] = ratio(df["P21"], df["P2"])
    out["lpc_pr"] = ratio(df["P24"], df["P21"])
    out["hpc_pr"] = ratio(df["Ps30"], df["P24"])
    out["eta_core"] = eta(ratio(df["P24"], df["P2"]), ratio(df["T24"], df["T2"]))
    out["eta_hpc"] = eta(out["hpc_pr"].to_numpy(), ratio(df["T30"], df["T24"]))
    out["burner_dp"] = 1.0 - ratio(df["P40"], df["Ps30"])
    out["lpt_tr"] = ratio(df["T50"], df["T48"])
    out["turb_pr"] = ratio(df["P40"], df["P50"])
    theta2 = ratio(df["T2"], 518.67)
    delta2 = ratio(df["P2"], 14.696)
    out["nf_corr"] = ratio(df["Nf"], np.sqrt(theta2))
    out["nc_corr"] = ratio(df["Nc"], np.sqrt(ratio(df["T24"], 518.67)))
    out["wf_corr"] = ratio(df["Wf"], delta2 * np.sqrt(theta2))
    out["spool_ratio"] = ratio(df["Nc"], df["Nf"])
    return out.astype(np.float32)


def scaled_w(w: np.ndarray) -> np.ndarray:
    c = np.array([W_CENTER[v][0] for v in W_VARS], dtype=np.float64)
    s = np.array([W_CENTER[v][1] for v in W_VARS], dtype=np.float64)
    return (np.asarray(w, dtype=np.float64) - c) / s


def poly_terms(n_vars: int, degree: int) -> list[tuple]:
    terms = [()]
    for d in range(1, degree + 1):
        terms += list(itertools.combinations_with_replacement(range(n_vars), d))
    return terms


def poly_features(z: np.ndarray, degree: int = 3) -> np.ndarray:
    z = np.asarray(z, dtype=np.float64)
    cols = [np.prod(z[:, list(t)], axis=1) if t else np.ones(len(z)) for t in poly_terms(z.shape[1], degree)]
    return np.column_stack(cols)


SUMMARY_PREFIXES = ("m_", "sd_", "mn_", "mx_", "ls_")   # flight mean, std, min, max, last value of each sensor


def aggregate_cycles(df: pd.DataFrame, degree: int = 3) -> pd.DataFrame:
    """Collapse 1 Hz rows into one row per (unit, flight cycle) - the flight-end cadence of pipeline.md 3.2.

    Per sensor: mean (m_), std (sd_), min (mn_), max (mx_) and last value (ls_). Also: physics-proxy means (p_),
    virtual-sensor means (v_, display only), flight descriptors, labels (rul, health parameters, hs) and the mean
    polynomial features of the flight condition (phi_). Because the healthy reference is linear in phi,
    mean(reference) = coef . mean(phi), which gives exact per-flight "healthy" sensor means without a second pass.
    Rows of a flight must be contiguous and in time order (checked by validate_split); flights are never mixed.
    """
    out = []
    vcols = [c for c in df.columns if c.startswith("v_")]
    tcols = [c for c in T_VARS if c in df.columns]
    for unit, part in df.groupby("unit", sort=True):
        cyc = part["cycle"].to_numpy()
        if np.any(np.diff(cyc) < 0):
            part = part.sort_values("cycle", kind="stable")
            cyc = part["cycle"].to_numpy()
        change = np.flatnonzero(np.diff(cyc) != 0) + 1
        starts = np.concatenate([[0], change]).astype(np.int64)
        ends = np.concatenate([change, [len(part)]]).astype(np.int64) - 1
        cnt = (ends - starts + 1).astype(np.float64)

        def rmean(a):
            a = np.asarray(a, dtype=np.float64).reshape(len(part), -1)
            ok = np.isfinite(a)
            s = np.add.reduceat(np.where(ok, a, 0.0), starts, axis=0)
            n = np.add.reduceat(ok.astype(np.float64), starts, axis=0)
            return s / np.where(n == 0, np.nan, n)

        def rmin(a):
            return np.minimum.reduceat(np.asarray(a, dtype=np.float64), starts, axis=0)

        def rmax(a):
            return np.maximum.reduceat(np.asarray(a, dtype=np.float64), starts, axis=0)

        rows = {"unit": np.full(len(starts), int(unit)), "cycle": cyc[starts].astype(int), "duration_s": cnt}
        if "Fc" in part:
            rows["fc"] = part["Fc"].to_numpy()[starts].astype(int)
        if "hs" in part:
            rows["hs"] = rmin(part["hs"].to_numpy()).astype(int)
        if "RUL" in part:
            rows["rul"] = rmin(part["RUL"].to_numpy())
        w = part[W_VARS].to_numpy(dtype=np.float64)
        wm = rmean(w)
        for j, v in enumerate(W_VARS):
            rows[f"{v}_mean"] = wm[:, j]
        rows["alt_max"] = rmax(part["alt"].to_numpy())
        rows["Mach_max"] = rmax(part["Mach"].to_numpy())
        x = part[XS_VARS].to_numpy(dtype=np.float64)
        xm, x2 = rmean(x), rmean(x ** 2)
        xmin, xmax, xlast = rmin(x), rmax(x), x[ends]
        for j, s in enumerate(XS_VARS):
            rows[f"m_{s}"] = xm[:, j]
            rows[f"sd_{s}"] = np.sqrt(np.clip(x2[:, j] - xm[:, j] ** 2, 0, None))
            rows[f"mn_{s}"] = xmin[:, j]
            rows[f"mx_{s}"] = xmax[:, j]
            rows[f"ls_{s}"] = xlast[:, j]
        pm = rmean(physics_1hz(part).to_numpy())
        for j, p in enumerate(PROXIES):
            rows[f"p_{p}"] = pm[:, j]
        if vcols:
            vm = rmean(part[vcols].to_numpy())
            for j, c in enumerate(vcols):
                rows[c] = vm[:, j]
        if tcols:
            tm = rmean(part[tcols].to_numpy())
            for j, c in enumerate(tcols):
                rows[c] = tm[:, j]
        phim = rmean(poly_features(scaled_w(w), degree))
        for j in range(phim.shape[1]):
            rows[f"phi_{j}"] = phim[:, j]
        out.append(pd.DataFrame(rows))
    return pd.concat(out, ignore_index=True).sort_values(["unit", "cycle"]).reset_index(drop=True)


# ----------------------------------------------------------------------------------------------- Layer 2: physics
class HealthyReference:
    """Healthy-condition baseline g(W) (pipeline.md 3.4 / 4.5): each measured sensor as a ridge-regularised degree-3
    polynomial of the flight condition (alt, Mach, TRA, T2), fitted on healthy (hs == 1) rows of TRAINING engines only.
    residual = measured / healthy - 1 removes the effect of the flight condition."""

    def __init__(self, degree: int = 3, alpha: float = 1e-3, sensors=XS_VARS):
        self.degree, self.alpha, self.sensors = degree, alpha, list(sensors)
        self.coef = None            # (n_terms, n_sensors)
        self.fit_rmse = None

    def fit(self, w: np.ndarray, x: np.ndarray) -> "HealthyReference":
        phi = poly_features(scaled_w(w), self.degree)
        a = phi.T @ phi + self.alpha * np.eye(phi.shape[1])
        self.coef = np.linalg.solve(a, phi.T @ np.asarray(x, dtype=np.float64))
        pred = phi @ self.coef
        self.fit_rmse = (np.sqrt(np.mean((pred - x) ** 2, axis=0)) / np.abs(np.mean(x, axis=0))).tolist()
        return self

    def expected_from_phi(self, phi_mean: np.ndarray) -> np.ndarray:
        return np.asarray(phi_mean, dtype=np.float64) @ self.coef

    def predict(self, w: np.ndarray) -> np.ndarray:
        return poly_features(scaled_w(w), self.degree) @ self.coef

    def to_dict(self):
        return {"degree": self.degree, "alpha": self.alpha, "sensors": self.sensors,
                "coef": np.asarray(self.coef).tolist(), "fit_rel_rmse": self.fit_rmse, "w_center": W_CENTER}

    @classmethod
    def from_dict(cls, d):
        r = cls(d["degree"], d["alpha"], d["sensors"])
        r.coef = np.asarray(d["coef"], dtype=np.float64)
        r.fit_rmse = d.get("fit_rel_rmse")
        return r


def add_residuals(cyc: pd.DataFrame, ref: HealthyReference) -> pd.DataFrame:
    """Add ref_<sensor> (healthy-engine value) and res_<sensor> (relative residual, measured/healthy - 1)."""
    cyc = cyc.copy()
    phi_cols = [f"phi_{j}" for j in range(ref.coef.shape[0])]
    exp = ref.expected_from_phi(cyc[phi_cols].to_numpy())
    for j, s in enumerate(ref.sensors):
        cyc[f"ref_{s}"] = exp[:, j]
        cyc[f"res_{s}"] = cyc[f"m_{s}"] / np.where(np.abs(exp[:, j]) < 1e-9, np.nan, exp[:, j]) - 1.0
    return cyc


def residual_by_condition(cyc: pd.DataFrame, n_bins: int = 3) -> pd.DataFrame:
    """Healthy-phase residual mean/std per sensor by altitude band (inspect the baseline by operating range, 4.5)."""
    h = cyc[cyc["hs"] == 1] if "hs" in cyc else cyc
    bins = pd.qcut(h["alt_mean"], n_bins, duplicates="drop")
    rows = []
    for b, g in h.groupby(bins, observed=True):
        for s in XS_VARS:
            r = g[f"res_{s}"]
            rows.append({"altitude_band": str(b), "sensor": s, "mean_pct": 100 * float(r.mean()),
                         "std_pct": 100 * float(r.std()), "cycles": int(len(g))})
    return pd.DataFrame(rows)


# ----------------------------------------------------------------------------------------------- health labels / mapping
def component_damage(df: pd.DataFrame, prefix: str = "") -> pd.DataFrame:
    """Per-component damage = larger of |efficiency modifier| and |flow modifier|."""
    return pd.DataFrame({c: np.maximum(df[prefix + e].abs(), df[prefix + f].abs()) for c, (e, f) in COMP_THETA.items()},
                        index=df.index)


def fit_damage_reference(cyc_train: pd.DataFrame, q: float = 0.95, floor: float = 1e-3) -> dict:
    """Failure-level damage per component: q-quantile over TRAINING engines of that component's damage at end of
    life. Defines the health-index mapping HI = 1 - damage / d_ref (validated against the truth in the notebook)."""
    last = cyc_train.sort_values("cycle").groupby("unit").tail(1)
    dmg = component_damage(last)
    return {c: max(float(np.quantile(dmg[c], q)), floor) for c in COMPONENTS}


def d_ref_of(d_ref, comp: str) -> float:
    return float(d_ref[comp]) if isinstance(d_ref, dict) else float(d_ref)


def component_health(df: pd.DataFrame, d_ref) -> pd.DataFrame:
    """TRUE health index from the hidden parameters T (offline: validation of the mapping only)."""
    dmg = component_damage(df)
    return pd.DataFrame({f"hi_{c}": (1.0 - dmg[c] / d_ref_of(d_ref, c)).clip(0.0, 1.0) for c in COMPONENTS},
                        index=df.index)


def kf_component_health(kf_out: pd.DataFrame, d_ref) -> pd.DataFrame:
    """Health index from the EKF parameter estimates kf_<param> (online), with a 1-sigma from the dominating parameter."""
    out = {}
    for c, (e, f) in COMP_THETA.items():
        d = d_ref_of(d_ref, c)
        ae, af = kf_out[f"kf_{e}"].abs(), kf_out[f"kf_{f}"].abs()
        use_e = ae >= af
        out[f"kf_hi_{c}"] = (1.0 - np.maximum(ae, af) / d).clip(0, 1)
        out[f"kf_hi_{c}_sigma"] = np.where(use_e, kf_out[f"kf_{e}_sigma"], kf_out[f"kf_{f}_sigma"]) / d
    return pd.DataFrame(out, index=kf_out.index)


# ----------------------------------------------------------------------------------------------- Layer 3: EKF
SCHED_VARS = ["alt_mean", "Mach_mean", "TRA_mean"]


def schedule(cyc: pd.DataFrame) -> np.ndarray:
    """Operating-point variables of the influence coefficients: [1, alt, Mach, TRA] (fixed scaling)."""
    z = [np.ones(len(cyc))]
    for v, raw in zip(SCHED_VARS, ["alt", "Mach", "TRA"]):
        c, s = W_CENTER[raw]
        z.append((cyc[v].to_numpy(dtype=np.float64) - c) / s)
    return np.column_stack(z)


def robust_var(a: np.ndarray) -> np.ndarray:
    """Per-column robust variance (1.4826 * MAD)^2."""
    a = np.asarray(a, dtype=np.float64)
    med = np.nanmedian(a, axis=0)
    return (1.4826 * np.nanmedian(np.abs(a - med), axis=0)) ** 2


def unique_shares(h: np.ndarray) -> np.ndarray:
    """For each column of h (S, G): the share of its norm NOT explained by a least-squares fit on the other columns
    (1 = fully separable, ~0 = a combination of the others; a zero column scores 0)."""
    norms = np.linalg.norm(h, axis=0)
    u = np.ones(h.shape[1])
    if h.shape[1] == 1:
        return u
    for j in range(h.shape[1]):
        others = np.delete(h, j, axis=1)
        c, *_ = np.linalg.lstsq(others, h[:, j], rcond=None)
        u[j] = np.linalg.norm(h[:, j] - others @ c) / max(norms[j], 1e-300) if norms[j] > 0 else 0.0
    return u


def merge_states(hw: np.ndarray, theta: np.ndarray, merge_cos: float, min_unique: float = 0.0, active=None):
    """Observability-driven state reduction (pipeline.md 4.2 / 4.8 step 9 / 5.4).

    hw: (n_points, S, P) whitened influence matrices at sampled operating points; theta: (n, P) training values of the
    P candidate parameters; active: indices of the parameters to consider (default all; never-excited parameters are
    excluded by the caller). Two passes:
      1. pairwise: merge the two groups whose sensor signatures are most similar (median |cosine| over operating points)
         while that similarity is >= merge_cos;
      2. joint: while some group's signature is mostly a combination of the others (median unique share < min_unique),
         merge that group with the group it is most similar to.
    A merged state moves its members together along their dominant joint direction v (first right singular vector of
    theta[:, members], uncentred): theta_members ~= v * x. Returns (groups, loadings)."""
    groups = [[int(j)] for j in (range(hw.shape[2]) if active is None else active)]

    def loading(members):
        if len(members) == 1:
            return np.ones(1)
        _, _, vt = np.linalg.svd(theta[:, members], full_matrices=False)
        v = vt[0]
        return -v if (theta[:, members] @ v).mean() < 0 else v

    def columns():
        cols = np.stack([np.einsum("nsj,j->ns", hw[:, :, g], loading(g)) for g in groups], axis=2)   # (n, S, G)
        unit = cols / np.maximum(np.linalg.norm(cols, axis=1, keepdims=True), 1e-300)
        cos = np.median(np.abs(np.einsum("nsa,nsb->nab", unit, unit)), axis=0)
        np.fill_diagonal(cos, 0.0)
        return cols, cos

    while len(groups) > 1:                                          # 1. pairwise
        _, cos = columns()
        a, b = np.unravel_index(np.argmax(cos), cos.shape)
        if cos[a, b] < merge_cos:
            break
        groups[a] = sorted(groups[a] + groups[b])
        del groups[b]
    while len(groups) > 1 and min_unique > 0:                       # 2. joint collinearity
        cols, cos = columns()
        uniq = np.median([unique_shares(c) for c in cols], axis=0)
        g = int(np.argmin(uniq))
        if uniq[g] >= min_unique:
            break
        partner = int(np.argmax(cos[g]))
        a, b = min(g, partner), max(g, partner)
        groups[a] = sorted(groups[a] + groups[b])
        del groups[b]
    return groups, [loading(g) for g in groups]


class GasPathEKF:
    """Extended Kalman filter on a reduced-order, dataset-calibrated gas-path observation model (pipeline.md 4-5).

    Observation (per flight):  res = b(w) + B(w) x + v,  res = 14 relative sensor residuals vs the healthy baseline,
    w = operating point [1, alt, Mach, TRA], b and B linear in w (re-linearised at every operating point; the model is
    linear in x, so the EKF update is exact). State: random walk x_k = x_(k-1) + w_k, advanced once per FLIGHT.

    Calibration (training engines only; T used offline as allowed by 4.7):
      * b, B per candidate parameter: ridge regression of residuals on the 10 N-CMAPSS health parameters;
      * R: robust variance of healthy-phase residuals (hs == 1) after removing b(w);
      * states: observability analysis merges parameters whose sensor signatures are indistinguishable (merge_cos);
      * Q: variance of cycle-to-cycle state changes, scaled by q_scale tuned on VALIDATION engines (one-step-ahead
        predictive log-likelihood of the sensors - no labels); x0 / P0 from first flights.
    Outputs per flight: state estimate + sigma, parameter estimates kf_<param> (= A x) + sigma, prior innovations,
    per-sensor NIS and gating flags, joint NIS, and the smallest covariance eigenvalue (PSD check).
    """

    def __init__(self, sensors=XS_VARS, params=T_VARS, ridge: float = 1e-4, gate: float = 9.0, merge_cos: float = 0.95,
                 merge_unique: float = 0.15, min_excitation: float = 1e-6):
        self.sensors, self.params = list(sensors), list(params)
        self.ridge, self.gate, self.merge_cos = ridge, gate, merge_cos
        self.merge_unique, self.min_excitation = merge_unique, min_excitation
        self.b = self.H = self.H_params = self.R = self.Q0 = self.x0 = self.P0 = self.A = None
        self.offset, self.excluded = np.zeros(len(self.params)), []
        self.groups, self.state_names, self.q_scale = None, None, 1.0
        self.last_loglik = None

    # -------------------------------------------------------------- calibration
    @staticmethod
    def _design(z, th):
        return np.column_stack([z] + [th[:, [p]] * z for p in range(th.shape[1])])

    def _whitened(self, z_rows):
        sw = 1.0 / np.sqrt(self.R)
        return np.stack([np.einsum("pks,k->sp", self.H_params, zr) * sw[:, None] for zr in z_rows])   # (n, S, P)

    def fit(self, cyc: pd.DataFrame, cyc_val: pd.DataFrame | None = None, q_grid=(0.1, 0.3, 1.0, 3.0, 10.0),
            n_points: int = 200, seed: int = 0) -> "GasPathEKF":
        cyc = cyc.sort_values(["unit", "cycle"]).reset_index(drop=True)
        y = cyc[[f"res_{s}" for s in self.sensors]].to_numpy(dtype=np.float64)
        th = cyc[self.params].to_numpy(dtype=np.float64)
        z = schedule(cyc)
        x = self._design(z, th)
        coef = np.linalg.solve(x.T @ x + self.ridge * np.eye(x.shape[1]), x.T @ y)     # (K + P*K, S)
        k = z.shape[1]
        self.b = coef[:k]                                                              # (K, S)
        self.H_params = coef[k:].reshape(len(self.params), k, len(self.sensors))       # (P, K, S)
        base = y - z @ self.b
        healthy = cyc["hs"].to_numpy() == 1 if "hs" in cyc else np.zeros(len(cyc), dtype=bool)
        r = robust_var(base[healthy]) if healthy.sum() >= 30 else np.var(y - x @ coef, axis=0)
        self.R = np.maximum(np.nan_to_num(r, nan=0.0), 1e-12)
        rng = np.random.default_rng(seed)
        pts = rng.choice(len(z), size=min(n_points, len(z)), replace=False)
        excited = th.std(axis=0) > self.min_excitation     # parameters that never move in training are not states
        if not excited.any():
            raise ValueError("no health parameter varies in the training engines - nothing for the EKF to estimate")
        self.excluded = [t for t, e in zip(self.params, excited) if not e]
        self.offset = np.where(excited, 0.0, th.mean(axis=0))   # held at their (constant) training value
        self.groups, loads = merge_states(self._whitened(z[pts]), th, self.merge_cos, self.merge_unique,
                                          active=np.flatnonzero(excited))
        p, g = len(self.params), len(self.groups)
        self.A = np.zeros((p, g))
        for j, (members, v) in enumerate(zip(self.groups, loads)):
            self.A[members, j] = v
        self.state_names = ["+".join(self.params[m] for m in members) for members in self.groups]
        self.H = np.einsum("pg,pks->gks", self.A, self.H_params)                        # (G, K, S)
        states = th @ self.A                                                            # (n, G) training state values
        units = cyc["unit"].to_numpy()
        d = np.concatenate([np.diff(states[units == u], axis=0) for u in pd.unique(units)])
        self.Q0 = np.var(d, axis=0) + np.mean(d, axis=0) ** 2 + 1e-12
        first = np.stack([states[units == u][0] for u in pd.unique(units)])
        self.x0, self.P0 = first.mean(axis=0), first.var(axis=0) + 1e-8
        self.q_scale = 1.0
        if cyc_val is not None and len(cyc_val):
            ll = {}
            for q in q_grid:
                self.q_scale = float(q)
                self._filter(cyc_val)
                ll[q] = self.last_loglik
            self.q_scale = float(max(ll, key=ll.get))
            self.q_loglik = {str(k_): float(v) for k_, v in ll.items()}
        return self

    @property
    def Q(self):
        return self.q_scale * self.Q0

    # -------------------------------------------------------------- filtering
    def h_at(self, z_row):
        """Observation Jacobian B(w) (S, G) and bias b(w) (S,) at operating point z_row."""
        return np.einsum("gks,k->sg", self.H, z_row), z_row @ self.b

    def observe(self, z_row, x):
        """Predicted residuals h(w, x) = b(w) + B(w) x."""
        h, b = self.h_at(z_row)
        return b + h @ x

    def _filter(self, cyc: pd.DataFrame, res_override: np.ndarray | None = None) -> dict:
        cyc = cyc.reset_index(drop=True)
        res = cyc[[f"res_{s}" for s in self.sensors]].to_numpy(dtype=np.float64) if res_override is None \
            else np.asarray(res_override, dtype=np.float64)
        z = schedule(cyc)
        n, s, g = len(cyc), len(self.sensors), len(self.state_names)
        xs, sig, psig = np.zeros((n, g)), np.zeros((n, g)), np.zeros((n, len(self.params)))
        innov, nis = np.full((n, s), np.nan), np.full((n, s), np.nan)
        gated, nis_tot, min_eig = np.zeros((n, s), dtype=bool), np.full(n, np.nan), np.zeros(n)
        q, eye = self.Q, np.eye(g)
        ll, units = 0.0, cyc["unit"].to_numpy()
        for u in pd.unique(units):
            idx = np.flatnonzero(units == u)
            x, pm = self.x0.copy(), np.diag(self.P0)
            consec = np.zeros(s, dtype=int)
            for i in idx:
                pm = pm + np.diag(q)
                h, b = self.h_at(z[i])
                ok = np.isfinite(res[i])
                nu = res[i] - b - h @ x
                innov[i, ok] = nu[ok]
                if ok.any():
                    sfull = h[ok] @ pm @ h[ok].T + np.diag(self.R[ok])
                    try:
                        nis_tot[i] = float(nu[ok] @ np.linalg.solve(sfull, nu[ok]))
                    except np.linalg.LinAlgError:
                        pass
                for j in range(s):
                    if not ok[j]:
                        continue
                    hj = h[j]
                    v = res[i, j] - b[j] - hj @ x
                    ph = pm @ hj
                    sv = hj @ ph + self.R[j]
                    nis[i, j] = v * v / sv
                    if nis[i, j] > self.gate and consec[j] < 3:
                        gated[i, j] = True
                        consec[j] += 1
                        continue
                    consec[j] = 0
                    ll += -0.5 * (np.log(2 * np.pi * sv) + v * v / sv)
                    kg = ph / sv
                    x = x + kg * v
                    ikh = eye - np.outer(kg, hj)
                    pm = ikh @ pm @ ikh.T + self.R[j] * np.outer(kg, kg)            # Joseph form
                pm = 0.5 * (pm + pm.T)                                              # symmetrise
                xs[i], sig[i] = x, np.sqrt(np.clip(np.diag(pm), 0, None))
                psig[i] = np.sqrt(np.clip(np.einsum("pg,gh,ph->p", self.A, pm, self.A), 0, None))
                min_eig[i] = float(np.linalg.eigvalsh(pm)[0])
        self.last_loglik = ll / max(n, 1)
        return {"xs": xs, "sig": sig, "psig": psig, "innov": innov, "nis": nis, "gated": gated, "nis_tot": nis_tot,
                "min_eig": min_eig}

    def run(self, cyc: pd.DataFrame, res_override: np.ndarray | None = None) -> pd.DataFrame:
        """Filter every engine causally (rows sorted by unit, cycle)."""
        r = self._filter(cyc, res_override)
        theta = r["xs"] @ self.A.T + self.offset
        out = {}
        for j, name in enumerate(self.state_names):
            out[f"ekf_{j}"] = r["xs"][:, j]
            out[f"ekf_{j}_sigma"] = r["sig"][:, j]
        for k_, t in enumerate(self.params):
            out[f"kf_{t}"] = theta[:, k_]
            out[f"kf_{t}_sigma"] = r["psig"][:, k_]
        for j, sname in enumerate(self.sensors):
            out[f"innov_{sname}"] = r["innov"][:, j]
            out[f"nis_{sname}"] = r["nis"][:, j]
            out[f"gated_{sname}"] = r["gated"][:, j]
        out["nis_total"] = r["nis_tot"]
        out["ekf_min_eig"] = r["min_eig"]
        return pd.DataFrame(out)

    def diagnostics(self, cyc: pd.DataFrame, n_points: int = 200, seed: int = 0, weak: float | None = None) -> dict:
        """Observability report (pipeline.md 5.4). Unique share = part of a whitened sensor signature NOT explained by
        the other columns (1 = separable, ~0 = collinear), for the raw parameters and for the STATES actually estimated.
        A component is identifiable if it was excited in training, none of its states is merged with another
        component's parameters, and its state(s) are separable (share >= `weak`, default merge_unique). Merging a
        component's own efficiency and flow into one state is the intended remedy and keeps it identifiable."""
        weak = self.merge_unique if weak is None else weak
        z = schedule(cyc)
        rng = np.random.default_rng(seed)
        pts = z[rng.choice(len(z), size=min(n_points, len(z)), replace=False)]
        hw = self._whitened(pts)                                                      # (n, S, P)
        active = [k for k, t in enumerate(self.params) if t not in self.excluded]
        uniq_param = np.full(len(self.params), np.nan)
        uniq_param[active] = np.median([unique_shares(h[:, active]) for h in hw], axis=0)
        hs_ = np.einsum("nsp,pg->nsg", hw, self.A)                                    # whitened STATE Jacobians
        uniq_state = np.median([unique_shares(h) for h in hs_], axis=0)
        sv = np.median([np.linalg.svd(h, compute_uv=False) for h in hs_], axis=0)
        comp = {}
        for c, (e, f) in COMP_THETA.items():
            ie, if_ = self.params.index(e), self.params.index(f)
            states = [g for g, grp in enumerate(self.groups) if ie in grp or if_ in grp]
            if not states:
                comp[c] = {"merged_with": [], "states": [], "excited": False, "own_parameters_merged": False,
                           "min_unique_share": None, "identifiable": False,
                           "reason": "no degradation of this component in the training engines"}
                continue
            members = {m for g in states for m in self.groups[g]}
            others = sorted({cc for cc, (e2, f2) in COMP_THETA.items() if cc != c and
                             (self.params.index(e2) in members or self.params.index(f2) in members)})
            m = float(min(uniq_state[g] for g in states))
            ident = bool(not others and m >= weak)
            reason = "ok" if ident else (f"indistinguishable from {', '.join(others)}" if others
                                         else "sensor signature too close to other states")
            comp[c] = {"merged_with": others, "states": [self.state_names[g] for g in states], "excited": True,
                       "own_parameters_merged": len(states) == 1 and len(members & {ie, if_}) == 2,
                       "min_unique_share": m, "identifiable": ident, "reason": reason}
        return {"unique_share_per_parameter": {t: (None if np.isnan(v) else float(v)) for t, v in zip(self.params, uniq_param)},
                "unique_share_per_state": {n: float(v) for n, v in zip(self.state_names, uniq_state)},
                "excluded_not_excited": list(self.excluded), "states": self.state_names,
                "state_singular_values_median": [float(v) for v in sv],
                "state_condition_number_median": float(sv[0] / max(sv[-1], 1e-300)),
                "components": comp, "merge_cos": self.merge_cos, "merge_unique": self.merge_unique,
                "q_scale": self.q_scale}

    def to_dict(self):
        return {"sensors": self.sensors, "params": self.params, "ridge": self.ridge, "gate": self.gate,
                "merge_cos": self.merge_cos, "merge_unique": self.merge_unique, "min_excitation": self.min_excitation,
                "offset": np.asarray(self.offset).tolist(), "excluded": list(self.excluded), "sched": SCHED_VARS, "groups": self.groups, "state_names": self.state_names,
                "A": self.A.tolist(), "b": self.b.tolist(), "H": self.H.tolist(), "H_params": self.H_params.tolist(),
                "R": self.R.tolist(), "Q0": self.Q0.tolist(), "q_scale": self.q_scale, "x0": self.x0.tolist(),
                "P0": self.P0.tolist()}

    @classmethod
    def from_dict(cls, d):
        k = cls(d["sensors"], d["params"], d["ridge"], d["gate"], d["merge_cos"], d.get("merge_unique", 0.15),
                d.get("min_excitation", 1e-6))
        k.offset = np.asarray(d.get("offset", np.zeros(len(d["params"]))), dtype=np.float64)
        k.excluded = list(d.get("excluded", []))
        for a in ("A", "b", "H", "H_params", "R", "Q0", "x0", "P0"):
            setattr(k, a, np.asarray(d[a], dtype=np.float64))
        k.groups, k.state_names, k.q_scale = d["groups"], d["state_names"], float(d["q_scale"])
        return k


def run_layers(cyc: pd.DataFrame, ref: HealthyReference, ekf: GasPathEKF, d_ref) -> pd.DataFrame:
    """Per-flight table -> residuals (Layer 2) -> EKF states, innovations and health mapping (Layer 3)."""
    cyc = add_residuals(cyc.sort_values(["unit", "cycle"]).reset_index(drop=True), ref)
    k = ekf.run(cyc)
    return pd.concat([cyc, k, kf_component_health(k, d_ref)], axis=1)


# ----------------------------------------------------------------------------------------------- Layer 4: features
FLIGHT_COLS = ["alt_mean", "Mach_mean", "TRA_mean", "T2_mean", "alt_max", "Mach_max", "duration_s", "fc"]
EXPOSURE_COLS = ["cycle", "exposure_h", "hist_len"]
WINDOW_STATS = ("rm", "rs", "rmin", "rmax", "slope")      # trailing-window mean, std, min, max, slope (6.2)
FORBIDDEN_FEATURES = {"rul", "unit", "unit_local", "hs", "dataset", "split", "cycle_id"}


def sensor_summary_cols() -> list[str]:
    return [f"{p}{s}" for p in SUMMARY_PREFIXES for s in XS_VARS]


def default_feature_columns(n_states: int):
    """Feature blocks of pipeline.md 6.1: (temporal bases, static columns)."""
    temporal = [f"res_{s}" for s in XS_VARS] + [f"p_{p}" for p in PROXIES] + [f"ekf_{j}" for j in range(n_states)]
    static = sensor_summary_cols() + [f"ekf_{j}_sigma" for j in range(n_states)] + [f"innov_{s}" for s in XS_VARS]
    return temporal, static


def feature_group(name: str) -> str:
    """Block of a feature: raw | res | phys | ekf | innov | cond (flight condition / exposure)."""
    base = name.split("__")[0]
    if base in FLIGHT_COLS or base in EXPOSURE_COLS:
        return "cond"
    for pre, grp in (("res_", "res"), ("p_", "phys"), ("ekf_", "ekf"), ("innov_", "innov")):
        if base.startswith(pre):
            return grp
    if base.startswith(SUMMARY_PREFIXES):
        return "raw"
    return "other"


def is_window_feature(name: str) -> bool:
    return "__" in name


def source_of(feature: str) -> str:
    """Physical source of a model feature (for grouping SHAP values)."""
    return feature.split("__")[0]


def audit_features(names) -> list[str]:
    """Leakage audit (6.3): no target, hidden parameters, identifiers or health flags among the model inputs."""
    bad = [n for n in names if n.split("__")[0] in FORBIDDEN_FEATURES or n.split("__")[0] in T_VARS
           or n.startswith(("hi_", "truth_", "kf_hi_", "rul"))]
    if bad:
        raise ValueError(f"leaking features: {bad}")
    return list(names)


class CausalFeaturizer:
    """Causal per-engine features (pipeline.md 6): current values of all blocks, trailing-window mean / std / min / max
    / slope of the temporal bases over `roll` flights, flight condition and operating exposure (cycle count, cumulative
    flight hours). Minimum-history rule: windows use the flights available so far (std = 0 and slope = 0 with fewer
    than 2 flights; `hist_len` tells the model how many). Missing values are filled with training medians."""

    def __init__(self, temporal_cols, static_cols, roll: int = 5):
        self.temporal_cols, self.static_cols, self.roll = list(temporal_cols), list(static_cols), int(roll)
        self.fill: dict = {}
        self.feature_names = [c for b in self.temporal_cols for c in [b] + [f"{b}__{st}" for st in WINDOW_STATS]] + \
            self.static_cols + FLIGHT_COLS + EXPOSURE_COLS
        audit_features(self.feature_names)

    def _build(self, cyc: pd.DataFrame) -> pd.DataFrame:
        cyc = cyc.sort_values(["unit", "cycle"]).reset_index(drop=True)
        units = cyc["unit"].to_numpy()
        a = cyc[self.temporal_cols].to_numpy(dtype=np.float64)
        w = self.roll
        stats = {st: np.zeros_like(a) for st in WINDOW_STATS}
        hist = np.zeros(len(cyc))
        expo = np.zeros(len(cyc))
        for u in pd.unique(units):
            idx = np.flatnonzero(units == u)
            g = pd.DataFrame(a[idx])
            roll = g.rolling(w, min_periods=1)
            stats["rm"][idx] = roll.mean().to_numpy()
            stats["rs"][idx] = roll.std(ddof=0).to_numpy()
            stats["rmin"][idx] = roll.min().to_numpy()
            stats["rmax"][idx] = roll.max().to_numpy()
            x = a[idx]
            sl = np.zeros_like(x)
            for i in range(len(idx)):
                win = x[max(0, i - w + 1): i + 1]
                if len(win) >= 2:
                    t = np.arange(len(win)) - (len(win) - 1) / 2.0
                    sl[i] = (t[:, None] * (win - np.nanmean(win, axis=0))).sum(axis=0) / (t ** 2).sum()
            stats["slope"][idx] = sl
            hist[idx] = np.minimum(np.arange(1, len(idx) + 1), w)
            expo[idx] = np.cumsum(cyc["duration_s"].to_numpy(dtype=np.float64)[idx]) / 3600.0
        out = {"unit": units, "cycle_id": cyc["cycle"].to_numpy()}
        for j, b in enumerate(self.temporal_cols):
            out[b] = a[:, j]
            for st in WINDOW_STATS:
                out[f"{b}__{st}"] = stats[st][:, j]
        for c in self.static_cols + FLIGHT_COLS:
            out[c] = cyc[c].to_numpy(dtype=np.float64) if c in cyc else np.full(len(cyc), np.nan)
        out["cycle"] = cyc["cycle"].to_numpy(dtype=np.float64)
        out["exposure_h"], out["hist_len"] = expo, hist
        return pd.DataFrame(out)

    def fit(self, cyc_train: pd.DataFrame) -> "CausalFeaturizer":
        f = self._build(cyc_train)[self.feature_names]
        self.fill = {c: float(v) if np.isfinite(v) else 0.0 for c, v in f.median().items()}
        return self

    def transform(self, cyc: pd.DataFrame) -> pd.DataFrame:
        out = self._build(cyc)
        if self.fill:
            out[self.feature_names] = out[self.feature_names].fillna(self.fill)
        return out

    def to_dict(self):
        return {"temporal_cols": self.temporal_cols, "static_cols": self.static_cols, "roll": self.roll,
                "fill": self.fill, "feature_names": self.feature_names}

    @classmethod
    def from_dict(cls, d):
        f = cls(d["temporal_cols"], d["static_cols"], d["roll"])
        f.fill = d.get("fill", {})
        return f


# ----------------------------------------------------------------------------------------------- robustness test helper
def inject_sensor_fault(cyc: pd.DataFrame, ref: HealthyReference, sensor: str, rel_bias: float,
                        start_frac: float = 0.5, every: int | None = None) -> pd.DataFrame:
    """Copy of a per-flight table with a fault in sensor `sensor`: from `start_frac` of each engine's life its flight
    mean (and min/max/last) is scaled by (1 + rel_bias) - every flight (persistent bias) or every `every`-th flight
    (spikes). Residuals are recomputed; physics proxies are left untouched (a conservative test). Adds `faulted`.
    Used on REAL test engines only to test robustness - nothing is fitted on it."""
    out = cyc.sort_values(["unit", "cycle"]).reset_index(drop=True)
    pos = out.groupby("unit").cumcount().to_numpy()
    size = out.groupby("unit")["cycle"].transform("size").to_numpy()
    faulted = pos >= start_frac * size
    if every:
        faulted &= (pos % every == 0)
    for pre in ("m_", "mn_", "mx_", "ls_"):
        col = f"{pre}{sensor}"
        if col in out:
            out.loc[faulted, col] = out.loc[faulted, col] * (1.0 + rel_bias)
    out = add_residuals(out, ref)
    out["faulted"] = faulted
    return out

In [31]:
if "nc" not in globals():          # in the notebook the module cell defines everything at top level
    import types
    nc = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith("_")})

In [32]:
"""Four tree regressors + Ridge stacking for RUL, exactly as pipeline.md sections 7-10 describe.

* Base models: Random Forest, XGBoost, LightGBM, CatBoost with the spec's starting configurations; small grids tuned
  on VALIDATION engines only; XGBoost uses early stopping on the validation engines.
* Stacking: engine-grouped out-of-fold predictions on the TRAINING engines -> StandardScaler fitted on those OOF
  predictions only -> Ridge (alpha chosen on validation engines) -> base models refitted on the full training partition.
* Metrics: RMSE, MAE, R2 and the PHM08/C-MAPSS NASA score (late predictions penalised more: exp(d/10) - 1 for
  d = pred - true > 0, exp(-d/13) - 1 for d < 0).
* Uncertainty: an EMPIRICAL interval from validation-engine residual quantiles (section 9), clearly labelled.

Shared by the Colab notebook (inlined) and the backend (StackModel only needs numpy at inference).
"""
from __future__ import annotations

import numpy as np

DEFAULT_SEED = 42
BASE_NAMES = ["randomforest", "xgboost", "lightgbm", "catboost"]

SPEC_PARAMS = {        # pipeline.md 7.1-7.4 starting configurations (search starting points, not calibrated optima)
    "randomforest": dict(n_estimators=400, max_features=0.7, min_samples_leaf=2),
    "xgboost": dict(n_estimators=800, max_depth=6, learning_rate=0.03, subsample=0.8, colsample_bytree=0.8,
                    reg_lambda=5.0),
    "lightgbm": dict(n_estimators=800, learning_rate=0.03, num_leaves=31, max_depth=-1, subsample=0.8,
                     colsample_bytree=0.8, reg_lambda=5.0),
    "catboost": dict(iterations=800, depth=7, learning_rate=0.03, l2_leaf_reg=5.0),
}
TUNE_GRIDS = {         # small validation-engine grids around the starting points
    "randomforest": [dict(max_features=f, min_samples_leaf=m) for f in (0.5, 0.7) for m in (2, 5)],
    "xgboost": [dict(max_depth=d) for d in (4, 6, 8)],
    "lightgbm": [dict(num_leaves=n) for n in (15, 31, 63)],
    "catboost": [dict(depth=d) for d in (5, 7)],
}
EARLY_STOPPING_ROUNDS = 50


def make_model(name: str, params: dict | None = None, seed: int = DEFAULT_SEED, n_jobs: int = -1):
    p = {**SPEC_PARAMS[name], **(params or {})}
    if name == "randomforest":
        from sklearn.ensemble import RandomForestRegressor
        return RandomForestRegressor(**p, n_jobs=n_jobs, random_state=seed)
    if name == "xgboost":
        import xgboost as xgb
        return xgb.XGBRegressor(**p, objective="reg:squarederror", tree_method="hist", n_jobs=n_jobs, random_state=seed)
    if name == "lightgbm":
        import lightgbm as lgb
        # subsample only takes effect with subsample_freq > 0 (LightGBM semantics, see pipeline.md 7.3)
        return lgb.LGBMRegressor(**p, subsample_freq=1, n_jobs=n_jobs, random_state=seed, verbosity=-1)
    if name == "catboost":
        from catboost import CatBoostRegressor
        return CatBoostRegressor(**p, loss_function="RMSE", random_seed=seed, verbose=False, thread_count=n_jobs,
                                 allow_writing_files=False)
    raise ValueError(f"unknown model {name}")


# ----------------------------------------------------------------------------------------------- metrics
def rmse(pred, true) -> float:
    d = np.asarray(pred, dtype=np.float64) - np.asarray(true, dtype=np.float64)
    return float(np.sqrt(np.mean(d ** 2))) if len(d) else float("nan")


def mae(pred, true) -> float:
    d = np.asarray(pred, dtype=np.float64) - np.asarray(true, dtype=np.float64)
    return float(np.mean(np.abs(d))) if len(d) else float("nan")


def r2(pred, true) -> float:
    t = np.asarray(true, dtype=np.float64)
    tot = float(np.sum((t - t.mean()) ** 2)) if len(t) else 0.0
    return float(1.0 - np.sum((np.asarray(pred) - t) ** 2) / tot) if tot > 0 else float("nan")


def nasa_score(pred, true) -> np.ndarray:
    d = np.asarray(pred, dtype=np.float64) - np.asarray(true, dtype=np.float64)
    return np.where(d < 0, np.exp(-d / 13.0) - 1.0, np.exp(d / 10.0) - 1.0)


def report(pred, true) -> dict:
    s = nasa_score(pred, true)
    return {"rmse": rmse(pred, true), "mae": mae(pred, true), "r2": r2(pred, true),
            "nasa_score_sum": float(s.sum()), "nasa_score_mean": float(s.mean()) if len(s) else float("nan"),
            "n": int(len(np.asarray(true)))}


# ----------------------------------------------------------------------------------------------- engine splits
def split_engines(units, strata=None, val_fraction: float = 0.2, seed: int = DEFAULT_SEED):
    """Split complete engine trajectories into (train_units, val_units) with a fixed seed. With `strata`
    ({unit: subset}), the split is done per subset so every subset is represented in validation (if it has >= 2)."""
    uniq = np.unique(np.asarray(units))
    rng = np.random.default_rng(seed)
    strata = strata or {u: 0 for u in uniq}
    val = []
    for s in sorted({strata[u] for u in uniq}, key=str):
        members = np.array(sorted(u for u in uniq if strata[u] == s))
        if len(members) < 2:
            continue
        k = max(1, int(round(val_fraction * len(members))))
        val += list(rng.choice(members, size=k, replace=False))
    val = np.array(sorted(val))
    return np.array(sorted(set(uniq) - set(val))), val


def engine_folds(groups, n_folds: int = 5):
    """GroupKFold by engine over the rows of the training partition (no shuffling of correlated samples)."""
    from sklearn.model_selection import GroupKFold
    groups = np.asarray(groups)
    k = min(n_folds, len(np.unique(groups)))
    return list(GroupKFold(k).split(np.zeros(len(groups)), groups=groups))


# ----------------------------------------------------------------------------------------------- tuning + base models
def tune_model(name, x_tr, y_tr, x_val, y_val, grid=None, factory=make_model):
    """Pick the grid point with the lowest validation-engine RMSE. The boosted models (XGBoost, LightGBM, CatBoost)
    early-stop on the validation engines (pipeline.md 7.2-7.4: tune the number of iterations on validation engines);
    the chosen number of trees is frozen into the returned params, so later refits (out-of-fold, final, ablation) use
    it without early stopping. Returns (best_params, table)."""
    grid = grid if grid is not None else TUNE_GRIDS[name]
    table, best, best_score = [], None, np.inf
    for cand in grid:
        params = dict(cand)
        if name == "xgboost":
            m = factory(name, {**params, "early_stopping_rounds": EARLY_STOPPING_ROUNDS})
            m.fit(x_tr, y_tr, eval_set=[(x_val, y_val)], verbose=False)
            params["n_estimators"] = int(m.best_iteration) + 1
        elif name == "lightgbm":
            import lightgbm as lgb
            m = factory(name, params)
            m.fit(x_tr, y_tr, eval_set=[(x_val, y_val)],
                  callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False)])
            params["n_estimators"] = max(1, int(m.best_iteration_ or m.n_estimators))
        elif name == "catboost":
            m = factory(name, {**params, "early_stopping_rounds": EARLY_STOPPING_ROUNDS})
            m.fit(x_tr, y_tr, eval_set=(x_val, y_val))
            params["iterations"] = int(m.get_best_iteration()) + 1
        else:
            m = factory(name, params)
            m.fit(x_tr, y_tr)
        score = rmse(m.predict(x_val), y_val)
        table.append({"model": name, "params": params, "val_rmse": score})
        if score < best_score:
            best, best_score = params, score
    return best, table


def oof_predictions(names, params, x, y, folds, factory=make_model) -> np.ndarray:
    """(n, len(names)) out-of-fold predictions: every row is predicted by models that never saw its engine."""
    out = np.full((len(y), len(names)), np.nan)
    for k, name in enumerate(names):
        for tr, te in folds:
            m = factory(name, params.get(name))
            m.fit(x.iloc[tr] if hasattr(x, "iloc") else x[tr], np.asarray(y)[tr])
            out[te, k] = m.predict(x.iloc[te] if hasattr(x, "iloc") else x[te])
    return out


def fit_base_models(names, params, x, y, factory=make_model) -> dict:
    """Refit each base model on the full training partition."""
    models = {}
    for name in names:
        m = factory(name, params.get(name))
        m.fit(x, np.asarray(y))
        models[name] = m
    return models


def predict_bases(models: dict, names, x) -> np.ndarray:
    return np.column_stack([np.asarray(models[n].predict(x), dtype=np.float64).reshape(-1) for n in names])


# ----------------------------------------------------------------------------------------------- Ridge meta-learner
class StackModel:
    """RUL = intercept + sum_i coef_i * (p_i - mean_i) / scale_i  (scaler fitted on out-of-fold predictions only)."""

    def __init__(self, names, mean, scale, coef, intercept, alpha=None, diagnostics=None):
        self.names = list(names)
        self.mean, self.scale = np.asarray(mean, dtype=np.float64), np.asarray(scale, dtype=np.float64)
        self.coef, self.intercept = np.asarray(coef, dtype=np.float64), float(intercept)
        self.alpha, self.diagnostics = alpha, diagnostics or {}

    def transform(self, base: np.ndarray) -> np.ndarray:
        return (np.asarray(base, dtype=np.float64) - self.mean) / self.scale

    def predict(self, base: np.ndarray) -> np.ndarray:
        return self.transform(base) @ self.coef + self.intercept

    def contributions(self, base: np.ndarray) -> np.ndarray:
        """coefficient_i * scaled_base_prediction_i per row (pipeline.md 11.2); the intercept is separate."""
        return self.transform(base) * self.coef

    def effective_weights(self) -> np.ndarray:
        """d RUL / d p_i (used to weight per-model SHAP values)."""
        return self.coef / self.scale

    def to_dict(self):
        return {"names": self.names, "mean": self.mean.tolist(), "scale": self.scale.tolist(),
                "coef": self.coef.tolist(), "intercept": self.intercept, "alpha": self.alpha,
                "diagnostics": self.diagnostics}

    @classmethod
    def from_dict(cls, d):
        return cls(d["names"], d["mean"], d["scale"], d["coef"], d["intercept"], d.get("alpha"), d.get("diagnostics"))


def fit_stack(oof: np.ndarray, y, names, val_base=None, y_val=None,
              alphas=(0.01, 0.1, 1.0, 10.0, 100.0, 1000.0), alpha: float | None = None) -> StackModel:
    """Scaler on the OOF matrix, Ridge on scaled OOF predictions; alpha by validation-engine RMSE (or fixed via
    `alpha`). Coefficients are not constrained to be positive or to sum to one (8.3)."""
    from sklearn.linear_model import Ridge

    y = np.asarray(y, dtype=np.float64)
    mean = oof.mean(axis=0)
    scale = oof.std(axis=0)
    scale = np.where(scale < 1e-12, 1.0, scale)
    z = (oof - mean) / scale
    table = {}
    if alpha is not None:
        pass
    elif val_base is not None and y_val is not None and len(y_val):
        zv = (np.asarray(val_base) - mean) / scale
        for a in alphas:
            table[a] = rmse(Ridge(alpha=a).fit(z, y).predict(zv), y_val)
        alpha = min(table, key=table.get)
    else:
        alpha = 1.0
    m = Ridge(alpha=alpha).fit(z, y)
    resid = y - m.predict(z)
    diag = {"alpha_val_rmse": {str(k): v for k, v in table.items()},
            "base_prediction_correlation": np.corrcoef(oof.T).tolist() if oof.shape[1] > 1 else [[1.0]],
            "oof_rmse_per_base": {n: rmse(oof[:, i], y) for i, n in enumerate(names)},
            "meta_residual_mean": float(resid.mean()), "meta_residual_std": float(resid.std())}
    return StackModel(names, mean, scale, m.coef_, m.intercept_, alpha, diag)


def cv_stack_predictions(oof: np.ndarray, y, folds, names, alpha: float) -> np.ndarray:
    """Engine-held-out predictions of the WHOLE stack on the training engines: for each fold, the scaler + Ridge are
    refitted on the other folds' out-of-fold predictions and applied to this fold. Used as extra (slightly pessimistic,
    because each base model saw fewer engines) residuals for the empirical interval."""
    y = np.asarray(y, dtype=np.float64)
    out = np.full(len(y), np.nan)
    for tr, te in folds:
        out[te] = fit_stack(oof[tr], y[tr], names, alpha=alpha).predict(oof[te])
    return out


# ----------------------------------------------------------------------------------------------- uncertainty
RUL_BAND_EDGES = (20.0, 60.0)      # predicted-RUL ranges of the interval: <= 20, (20, 60], > 60


def banded_interval(pred, residuals, level: float = 0.90, edges=RUL_BAND_EDGES, min_n: int = 30) -> dict:
    """Empirical interval PER PREDICTED-RUL RANGE (pipeline.md 9: error by degradation group): quantiles of
    residuals (true - predicted) of the rows whose prediction falls in each range. Ranges with fewer than `min_n`
    residuals use the pooled quantiles. Bands are defined on the prediction because the true RUL is unknown online."""
    pred, r = np.asarray(pred, dtype=np.float64), np.asarray(residuals, dtype=np.float64)
    low, high = empirical_interval(r, level)
    bounds = [-np.inf, *edges, np.inf]
    bands = []
    for a, b in zip(bounds[:-1], bounds[1:]):
        m = (pred > a) & (pred <= b)
        lo, hi, src = (*empirical_interval(r[m], level), "band") if m.sum() >= min_n else (low, high, "pooled")
        bands.append({"pred_above": None if np.isinf(a) else float(a), "pred_up_to": None if np.isinf(b) else float(b),
                      "low": lo, "high": hi, "n": int(m.sum()), "source": src})
    return {"level": level, "low": low, "high": high, "bands": bands}


def apply_interval(pred, interval: dict):
    """(lower, upper) RUL bounds for each prediction, clipped at 0; uses the band of each prediction if present."""
    pred = np.asarray(pred, dtype=np.float64)
    lo, hi = np.full(len(pred), interval["low"]), np.full(len(pred), interval["high"])
    for band in interval.get("bands") or []:
        a = -np.inf if band["pred_above"] is None else band["pred_above"]
        b = np.inf if band["pred_up_to"] is None else band["pred_up_to"]
        m = (pred > a) & (pred <= b)
        lo[m], hi[m] = band["low"], band["high"]
    return np.clip(pred + lo, 0.0, None), np.clip(pred + hi, 0.0, None)


def empirical_interval(residuals, level: float = 0.90):
    """(low, high) offsets from validation residual quantiles (true - predicted): interval = prediction + offsets.
    An empirical prediction interval, not a calibrated probability (pipeline.md 9)."""
    r = np.asarray(residuals, dtype=np.float64)
    a = (1.0 - level) / 2.0
    return float(np.quantile(r, a)), float(np.quantile(r, 1.0 - a))


def error_by_group(pred, true, labels) -> list[dict]:
    """RMSE / MAE per group (engine, subset, flight class, degradation phase...)."""
    pred, true, labels = np.asarray(pred), np.asarray(true), np.asarray(labels)
    rows = []
    for g in sorted(np.unique(labels), key=str):
        m = labels == g
        rows.append({"group": g.item() if hasattr(g, "item") else g, "n": int(m.sum()),
                     "rmse": rmse(pred[m], true[m]), "mae": mae(pred[m], true[m]),
                     "bias": float(np.mean(pred[m] - true[m]))})
    return rows


def rul_phase(rul) -> np.ndarray:
    """Degradation phase label from the true RUL (for reporting only)."""
    r = np.asarray(rul, dtype=np.float64)
    return np.where(r <= 20, "end_of_life (RUL<=20)", np.where(r <= 60, "mid (20<RUL<=60)", "early (RUL>60)"))


# ----------------------------------------------------------------------------------------------- forecasting
# EXTENSION beyond pipeline.md (which defines RUL as the only ML target): direct multi-horizon forecasts of component
# health. Target = true health h flights later MINUS the current EKF health estimate (so a model that learns nothing
# reproduces the "no change" baseline). Future labels come from the hidden parameters T and are used as targets only.
def future_values(units, cycles, values, h: int) -> np.ndarray:
    """Value of the same engine h flights later (matched by cycle number); NaN where the engine ends first.
    Rows must be sorted by (unit, cycle)."""
    units, cycles = np.asarray(units), np.asarray(cycles, dtype=np.float64)
    values = np.asarray(values, dtype=np.float64)
    n = len(values)
    out = np.full(n, np.nan)
    if n == 0:
        return out
    j = np.minimum(np.arange(n) + int(h), n - 1)
    ok = (np.arange(n) + int(h) < n) & (units[j] == units) & (cycles[j] - cycles == h)
    out[ok] = values[j[ok]]
    return out


def trend_projection(units, cycles, values, h: int, window: int = 5) -> np.ndarray:
    """Causal straight-line baseline: per engine, least-squares slope (per flight) of the last `window` values,
    projected h flights ahead from the current value and clipped to [0, 1]."""
    units, values = np.asarray(units), np.asarray(values, dtype=np.float64)
    out = values.copy()
    for u in np.unique(units):
        idx = np.flatnonzero(units == u)
        v = values[idx]
        for i in range(len(idx)):
            win = v[max(0, i - window + 1): i + 1]
            if len(win) >= 2:
                t = np.arange(len(win)) - (len(win) - 1) / 2.0
                slope = (t * (win - win.mean())).sum() / (t ** 2).sum()
                out[idx[i]] = v[i] + slope * h
    return np.clip(out, 0.0, 1.0)


def forecast_metrics(pred, truth, persistence, trend, current_truth, degraded_below: float = 0.9) -> dict:
    """RMSE/MAE of the forecast and of both baselines on all rows and on DEGRADED rows (current true health below
    `degraded_below`; selected on the present, never on the future outcome). skill = 1 - RMSE/RMSE_persistence."""
    pred, truth = np.asarray(pred, dtype=np.float64), np.asarray(truth, dtype=np.float64)
    refs = {"forecast": pred, "persistence": np.asarray(persistence, dtype=np.float64),
            "trend": np.asarray(trend, dtype=np.float64)}
    cur = np.asarray(current_truth, dtype=np.float64)
    out = {}
    for part, m in (("all", np.ones(len(truth), dtype=bool)), ("degraded", cur < degraded_below)):
        row = {"n": int(m.sum())}
        for name, p in refs.items():
            row[f"{name}_rmse"] = rmse(p[m], truth[m]) if m.any() else None
            row[f"{name}_mae"] = mae(p[m], truth[m]) if m.any() else None
        base = row["persistence_rmse"]
        row["skill_vs_persistence"] = (1.0 - row["forecast_rmse"] / base) if base else None
        base = row["trend_rmse"]
        row["skill_vs_trend"] = (1.0 - row["forecast_rmse"] / base) if base else None
        out[part] = row
    return out

In [33]:
if "st" not in globals():
    import types
    st = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith("_")})

## 4. Pre-flight: files on Kaggle, sizes, disk

In [34]:
EXPECTED_BYTES = {                                            # sizes listed by Kaggle (checked 2026-10)
    "N-CMAPSS_DS01-005.h5": 2873351432, "N-CMAPSS_DS02-006.h5": 2450472504, "N-CMAPSS_DS03-012.h5": 3693395776,
    "N-CMAPSS_DS04.h5": 3752493952, "N-CMAPSS_DS05.h5": 2599166216, "N-CMAPSS_DS06.h5": 2549159720,
    "N-CMAPSS_DS07.h5": 2714714776, "N-CMAPSS_DS08a-009.h5": 3236762200, "N-CMAPSS_DS08c-008.h5": 2413078176,
    "N-CMAPSS_DS08d-010.h5": 2885034848}                      # DS08d-010 is 32 bytes short of its own HDF5 header


def remote_sizes():
    try:
        import urllib.request
        with urllib.request.urlopen(f"https://www.kaggle.com/api/v1/datasets/list/{DATASET_HANDLE}", timeout=30) as r:
            return {f["name"]: int(f["totalBytes"]) for f in json.load(r)["datasetFiles"]}
    except Exception as exc:
        print("could not list the Kaggle dataset (continuing with the built-in size table):", repr(exc))
        return {}


REMOTE = remote_sizes()
print(f"RUN={RUN}: {len(FILES)} file(s)")
for f_ in FILES:
    exp = REMOTE.get(f_) or EXPECTED_BYTES.get(f_)
    print(f"  {f_:26s} {exp / 1e9 if exp else float('nan'):5.2f} GB   on Kaggle: {f_ in REMOTE}")
free_gb = shutil.disk_usage(os.getcwd()).free / 1e9
print(f"free disk: {free_gb:.0f} GB (one file at a time needs ~4 GB)", "" if free_gb > 6 else "  <-- LOW")

RUN=full: 9 file(s)
  N-CMAPSS_DS01-005.h5        2.87 GB   on Kaggle: True
  N-CMAPSS_DS02-006.h5        2.45 GB   on Kaggle: True
  N-CMAPSS_DS03-012.h5        3.69 GB   on Kaggle: True
  N-CMAPSS_DS04.h5            3.75 GB   on Kaggle: True
  N-CMAPSS_DS05.h5            2.60 GB   on Kaggle: True
  N-CMAPSS_DS06.h5            2.55 GB   on Kaggle: True
  N-CMAPSS_DS07.h5            2.71 GB   on Kaggle: True
  N-CMAPSS_DS08a-009.h5       3.24 GB   on Kaggle: True
  N-CMAPSS_DS08c-008.h5       2.41 GB   on Kaggle: True
free disk: 93 GB (one file at a time needs ~4 GB) 


## 5. Layer 1: download, schema inspection + validation, flight-end aggregation (file by file, cached)

In [35]:
SEARCH = [(DATA_DIR, True), ("/kaggle/input", True), (DRIVE_DIR if DRIVE_OK else None, True), ("/content", False),
          ("/content/drive/MyDrive", False), (os.getcwd(), False)]


def find_local(fname):
    for root, recursive in SEARCH:
        if not root or not os.path.isdir(root):
            continue
        pattern = os.path.join(root, "**", fname) if recursive else os.path.join(root, fname)
        hits = glob.glob(pattern, recursive=recursive)
        if hits:
            return hits[0]
    return None


def get_file(fname):
    """(path, downloaded). Local copies are never deleted; downloads are deleted after use if DELETE_AFTER_USE."""
    local = find_local(fname)
    if local:
        return local, False
    import kagglehub
    try:
        return kagglehub.dataset_download(DATASET_HANDLE, path=fname), True
    except Exception as exc:
        raise RuntimeError(
            f"Could not download {fname} from Kaggle ({exc!r}).\nAdd Colab secrets KAGGLE_USERNAME / KAGGLE_KEY "
            "(kaggle.com > Settings > API > Create New Token), run kagglehub.login(), or set NCMAPSS_DIR.") from exc


def file_fingerprint(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        h.update(fh.read(4_000_000))
    return {"bytes": os.path.getsize(path), "sha256_first_4MB": h.hexdigest()}


def cache_base(fname):
    if not (CACHE_DIR and CACHE_AGGREGATES):
        return None
    return os.path.join(CACHE_DIR, f"agg_{fname.replace('.h5', '')}_v{nc.AGG_VERSION}_h{HEALTHY_SAMPLES_PER_FILE}_s{SEED}")


def cache_load(base):
    if not base or not os.path.exists(base + ".meta.json"):
        return None
    try:
        with open(base + ".meta.json") as fh:
            meta = json.load(fh)
        z = np.load(base + ".healthy.npz")
        return meta, pd.read_parquet(base + ".cycles.parquet"), z["w"], z["x"], z["u"]
    except Exception as exc:
        print("  cache unreadable, recomputing:", repr(exc))
        return None


def cache_save(base, meta, cycles, hw, hx, hu):
    if not base:
        return
    try:
        cycles.to_parquet(base + ".cycles.parquet", index=False)
        np.savez(base + ".healthy.npz", w=hw, x=hx, u=hu)
        with open(base + ".meta.json", "w") as fh:                    # written last: marks the cache entry complete
            json.dump(meta, fh)
    except Exception as exc:
        print("  could not write the cache (continuing):", repr(exc))


def process_file(fname):
    """Download (if needed), inspect, validate and aggregate one file. Unit ids stay file-local here."""
    rng_f = np.random.default_rng([SEED, zlib.crc32(fname.encode())])    # same healthy sample with or without cache
    path, downloaded = get_file(fname)
    fp = file_fingerprint(path)
    expected = REMOTE.get(fname) or EXPECTED_BYTES.get(fname)
    if expected and fp["bytes"] != expected:
        print(f"  WARNING: {fname} has {fp['bytes']} bytes, Kaggle lists {expected}")
    h5_path, repaired = nc.repair_truncated_h5(path, os.getcwd())         # Kaggle's DS08d-010 is 32 bytes short
    if repaired:
        print(f"  NOTE: {fname} is shorter than its HDF5 header records (mirror defect); opened a zero-padded copy")
    schema = {"inspect": nc.inspect_h5(h5_path), "splits": {}}
    ds = fname.replace("N-CMAPSS_", "").replace(".h5", "")
    cyc_parts, hw, hx, hu, logs = [], [], [], [], []
    for split in ("dev", "test"):
        t1 = time.time()
        df = nc.load_h5(h5_path, split)
        rep = nc.validate_split(df, split)
        schema["splits"][split] = rep
        if rep["dropped_rows"] or rep["missing_values"] or not rep.get("rul_non_increasing_within_engine", True):
            print(f"  NOTE [{split}]: dropped {rep['dropped_rows']} corrupt rows; missing values {rep['missing_values']}; "
                  f"RUL non-increasing: {rep.get('rul_non_increasing_within_engine')}")
        agg = nc.aggregate_cycles(df)
        agg["split"], agg["dataset"], agg["unit_local"] = split, ds, agg["unit"]
        cyc_parts.append(agg)
        if split == "dev":
            h = df[df["hs"] == 1] if "hs" in df else df
            take = rng_f.choice(len(h), size=min(HEALTHY_SAMPLES_PER_FILE, len(h)), replace=False)
            hw.append(h[nc.W_VARS].to_numpy()[take])
            hx.append(h[nc.XS_VARS].to_numpy()[take])
            hu.append(h["unit"].to_numpy()[take])
            del h
        logs.append({"file": fname, "split": split, "rows_1hz": rep["rows_1hz"], "engines": rep["engines"],
                     "flights": int(len(agg)), "seconds": round(time.time() - t1, 1)})
        print("  ", logs[-1])
        del df
        gc.collect()
    if repaired:
        os.remove(repaired)
    if downloaded and DELETE_AFTER_USE:
        try:
            os.remove(path)
        except OSError:                             # read-only mount (e.g. /kaggle/input): nothing to free
            pass
    return pd.concat(cyc_parts, ignore_index=True), np.vstack(hw), np.vstack(hx), np.concatenate(hu), logs, fp, schema


stage("Layer 1: download + aggregate")
cycles, healthy_w, healthy_x, healthy_u, load_log, FINGERPRINTS, SCHEMA = [], [], [], [], [], {}, {}
for fi, fname in enumerate(FILES):
    t0 = time.time()
    base = cache_base(fname)
    hit = cache_load(base)
    if hit:
        meta, agg, hw, hx, hu = hit
        fp, logs, schema, src = meta["fingerprint"], meta["load_log"], meta["schema"], "cache"
    else:
        agg, hw, hx, hu, logs, fp, schema = process_file(fname)
        cache_save(base, {"fingerprint": fp, "load_log": logs, "schema": schema}, agg, hw, hx, hu)
        src = "downloaded"
    agg = agg.copy()
    agg["unit"] = (fi + 1) * 1000 + agg["unit_local"]               # composite engine id: unique across files
    cycles.append(agg)
    healthy_w.append(hw)
    healthy_x.append(hx)
    healthy_u.append((fi + 1) * 1000 + np.asarray(hu).astype(np.int64))
    FINGERPRINTS[fname], SCHEMA[fname] = fp, schema
    load_log.extend(logs)
    print(f"{fname}: {src}, {time.time() - t0:.0f}s")
cyc = pd.concat(cycles, ignore_index=True).sort_values(["unit", "cycle"]).reset_index(drop=True)
del cycles
assert not (set(cyc[cyc.split == "dev"].unit) & set(cyc[cyc.split == "test"].unit)), "dev/test engine overlap"
TIMINGS["download_and_aggregate_s"] = time.time() - T_START
print(f"{cyc.unit.nunique()} engines, {len(cyc)} flights")


=== Layer 1: download + aggregate   [0.1 min elapsed]
   {'file': 'N-CMAPSS_DS01-005.h5', 'split': 'dev', 'rows_1hz': 4906636, 'engines': 6, 'flights': 553, 'seconds': 40.6}
   {'file': 'N-CMAPSS_DS01-005.h5', 'split': 'test', 'rows_1hz': 2735232, 'engines': 4, 'flights': 341, 'seconds': 16.5}
N-CMAPSS_DS01-005.h5: downloaded, 58s
   {'file': 'N-CMAPSS_DS02-006.h5', 'split': 'dev', 'rows_1hz': 5263447, 'engines': 6, 'flights': 446, 'seconds': 31.5}
   {'file': 'N-CMAPSS_DS02-006.h5', 'split': 'test', 'rows_1hz': 1253743, 'engines': 3, 'flights': 202, 'seconds': 7.4}
N-CMAPSS_DS02-006.h5: downloaded, 40s
   {'file': 'N-CMAPSS_DS03-012.h5', 'split': 'dev', 'rows_1hz': 5571277, 'engines': 9, 'flights': 663, 'seconds': 33.5}
   {'file': 'N-CMAPSS_DS03-012.h5', 'split': 'test', 'rows_1hz': 4251560, 'engines': 6, 'flights': 438, 'seconds': 25.5}
N-CMAPSS_DS03-012.h5: downloaded, 60s
   {'file': 'N-CMAPSS_DS04.h5', 'split': 'dev', 'rows_1hz': 6377452, 'engines': 6, 'flights': 512, 'seconds':

## 6. Engine split: train / validation / test - BEFORE anything is fitted (pipeline.md rule 4, 2.4)
Test = NASA's official test engines (untouched until the final evaluation). Validation = a fixed-seed share of the
development engines, stratified by subset, used for tuning, early stopping, the Ridge alpha and the EKF process noise.

In [36]:
stage("Engine split")
groups = cyc["unit"].to_numpy()
subset_of = cyc.drop_duplicates("unit").set_index("unit")["dataset"].to_dict()
dev_units = np.unique(groups[(cyc.split == "dev").to_numpy()])
train_units, val_units = st.split_engines(dev_units, {u: subset_of[u] for u in dev_units}, VAL_FRACTION, SEED)
test_units = np.unique(groups[(cyc.split == "test").to_numpy()])
cyc["partition"] = np.where(np.isin(groups, train_units), "train", np.where(np.isin(groups, val_units), "val", "test"))
train_m, val_m, test_m = [(cyc.partition == p).to_numpy() for p in ("train", "val", "test")]
tr_idx, va_idx, te_idx = np.flatnonzero(train_m), np.flatnonzero(val_m), np.flatnonzero(test_m)
assert set(np.unique(groups[te_idx])) == set(test_units) and not set(train_units) & set(val_units)
folds = st.engine_folds(groups[tr_idx], N_FOLDS)                  # positions within tr_idx
print(pd.crosstab(cyc.drop_duplicates("unit")["dataset"], cyc.drop_duplicates("unit")["partition"]))
print(f"flights: train {len(tr_idx)}, validation {len(va_idx)}, test {len(te_idx)}; {len(folds)} engine-grouped folds")


=== Engine split   [7.4 min elapsed]
partition  test  train  val
dataset                    
DS01-005      4      5    1
DS02-006      3      5    1
DS03-012      6      7    2
DS04          4      5    1
DS05          4      5    1
DS06          4      5    1
DS07          4      5    1
DS08a-009     6      7    2
DS08c-008     4      5    1
flights: train 3716, validation 819, test 2938; 5 engine-grouped folds


## 7. Layer 2: healthy-condition baseline g(W) and residuals (cross-fitted for training engines)
g(W) is fitted on healthy (hs == 1) rows of TRAINING engines only. Training-engine residuals use a baseline fitted
without that engine's fold (pipeline.md 3.4), so they are as out-of-sample as validation/test residuals.

In [37]:
stage("Layer 2: healthy baseline")
t0 = time.time()
hw_all, hx_all, hu_all = np.vstack(healthy_w), np.vstack(healthy_x), np.concatenate(healthy_u)
del healthy_w, healthy_x, healthy_u
in_train = np.isin(hu_all, train_units)
ref = nc.HealthyReference().fit(hw_all[in_train], hx_all[in_train])          # the deployed baseline
print("healthy baseline relative RMSE per sensor (1 Hz, healthy training rows):")
print(pd.Series(ref.fit_rmse, index=nc.XS_VARS).map(lambda v: f"{100 * v:.3f}%").to_string())
cyc = nc.add_residuals(cyc, ref)
res_cols = [f"ref_{s}" for s in nc.XS_VARS] + [f"res_{s}" for s in nc.XS_VARS]
for a, b in folds:                                                            # cross-fitting for training rows
    rows = tr_idx[b]
    keep = in_train & ~np.isin(hu_all, np.unique(groups[rows]))
    sub = nc.add_residuals(cyc.iloc[rows], nc.HealthyReference().fit(hw_all[keep], hx_all[keep]))
    cyc.loc[cyc.index[rows], res_cols] = sub[res_cols].to_numpy()
del hw_all, hx_all, hu_all
print("\nhealthy-phase residuals of VALIDATION engines by altitude band (should be ~0 with small spread):")
res_band = nc.residual_by_condition(cyc[val_m])
print(res_band.pivot_table(index="sensor", columns="altitude_band", values="mean_pct").round(3).to_string())

# Health-index mapping (offline, from T): failure-level damage per component, on training engines only
D_REF = nc.fit_damage_reference(cyc[train_m])
cyc = pd.concat([cyc, nc.component_health(cyc, D_REF)], axis=1)               # TRUE health: validation only
print("\nfailure-level damage per component D_REF =", {c: round(v, 5) for c, v in D_REF.items()})
TIMINGS["layer2_s"] = time.time() - t0


=== Layer 2: healthy baseline   [7.4 min elapsed]
healthy baseline relative RMSE per sensor (1 Hz, healthy training rows):
T24     0.029%
T30     0.055%
T48     0.123%
T50     0.153%
P15     0.050%
P2      0.019%
P21     0.050%
P24     0.071%
Ps30    0.132%
P40     0.129%
P50     0.065%
Nf      0.058%
Nc      0.029%
Wf      0.227%

healthy-phase residuals of VALIDATION engines by altitude band (should be ~0 with small spread):
altitude_band  (15603.865, 24689.841]  (7366.343, 9888.502]  (9888.502, 15603.865]
sensor                                                                            
Nc                             -0.003                 0.009                 -0.001
Nf                              0.004                -0.002                  0.004
P15                            -0.000                 0.005                 -0.001
P2                             -0.000                -0.000                  0.000
P21                            -0.000                 0.005           

## 8. Layer 3: reduced-order gas-path model + EKF (states, covariance, innovations) and its diagnostics

In [38]:
stage("Layer 3: EKF")
t0 = time.time()
ekf = nc.GasPathEKF(gate=EKF_GATE, merge_cos=MERGE_COS, merge_unique=MERGE_UNIQUE).fit(cyc[train_m], cyc_val=cyc[val_m], q_grid=Q_GRID)
k_out = ekf.run(cyc)
cyc = pd.concat([cyc, k_out, nc.kf_component_health(k_out, D_REF)], axis=1)
OBS = ekf.diagnostics(cyc[train_m])
print(f"EKF states ({len(ekf.state_names)}): {ekf.state_names}")
print("not estimated (never moved in the training engines, held at their training value):", ekf.excluded or "none")
print(f"process-noise scale chosen on validation engines: {ekf.q_scale} (log-likelihood per flight "
      f"{getattr(ekf, 'q_loglik', {})})")
print(f"observability: median condition number of the whitened state Jacobian {OBS['state_condition_number_median']:.1f}")
print(pd.DataFrame(OBS["components"]).T.to_string())

v = cyc[val_m]
EKF_DIAG = {
    "mean_nis_per_sensor_val": {s: float(v[f"nis_{s}"].mean()) for s in nc.XS_VARS},
    "mean_joint_nis_val": float(v["nis_total"].mean()), "joint_nis_dof": len(nc.XS_VARS),
    "gated_share_val": float(v[[f"gated_{s}" for s in nc.XS_VARS]].to_numpy().mean()),
    "innovation_mean_val": {s: float(v[f"innov_{s}"].mean()) for s in nc.XS_VARS},
    "min_covariance_eigenvalue": float(cyc["ekf_min_eig"].min()),
    "efficiency_mods_improving_share": float((cyc[[f"kf_{t}" for t in nc.T_VARS if t.endswith("eff_mod")]] > 0.005)
                                             .to_numpy().mean()),
}
print(f"validation: mean joint NIS {EKF_DIAG['mean_joint_nis_val']:.1f} (expected ~{len(nc.XS_VARS)} if consistent), "
      f"gated share {EKF_DIAG['gated_share_val']:.2%}, min covariance eigenvalue {EKF_DIAG['min_covariance_eigenvalue']:.2e}")

# Validate the health-index mapping against the truth (T): validation engines decide reliability, test is reported
HEALTH_MAP, RELIABILITY = {}, {}
for c in nc.COMPONENTS:
    row = {}
    for part, m in (("val", val_m), ("test", test_m)):
        t, e = cyc.loc[m, f"hi_{c}"].to_numpy(), cyc.loc[m, f"kf_hi_{c}"].to_numpy()
        dg = t < 0.9
        row[part] = {"rmse": st.rmse(e, t), "mae": st.mae(e, t), "n_degraded": int(dg.sum()),
                     "rmse_degraded": st.rmse(e[dg], t[dg]) if dg.any() else None,
                     "corr": float(np.corrcoef(t, e)[0, 1]) if np.std(t) > 0 and np.std(e) > 0 else None}
    HEALTH_MAP[c] = row
    vd = row["val"]
    ident = OBS["components"][c]["identifiable"]
    if vd["n_degraded"] < RELIABLE_MIN_CYCLES:
        status = "insufficient_degradation_in_validation"
    else:
        status = "ok" if ident and vd["rmse_degraded"] <= RELIABLE_MAX_RMSE else "low"
    RELIABILITY[c] = {"status": status, "identifiable": ident, "merged_with": OBS["components"][c]["merged_with"],
                      "val_rmse_degraded_points": None if vd["rmse_degraded"] is None else 100 * vd["rmse_degraded"]}
print("\nhealth-index mapping vs truth (points out of 100) and reliability (decided on VALIDATION engines):")
print(pd.DataFrame({c: {"val RMSE degraded": None if r["val"]["rmse_degraded"] is None else 100 * r["val"]["rmse_degraded"],
                        "test RMSE degraded": None if r["test"]["rmse_degraded"] is None else 100 * r["test"]["rmse_degraded"],
                        "reliability": RELIABILITY[c]["status"]} for c, r in HEALTH_MAP.items()}).T.to_string())
TIMINGS["layer3_s"] = time.time() - t0


=== Layer 3: EKF   [7.6 min elapsed]
EKF states (8): ['fan_eff_mod+LPT_eff_mod', 'fan_flow_mod', 'LPC_eff_mod', 'LPC_flow_mod', 'HPC_eff_mod', 'HPC_flow_mod', 'HPT_eff_mod+LPT_flow_mod', 'HPT_flow_mod']
not estimated (never moved in the training engines, held at their training value): none
process-noise scale chosen on validation engines: 3.0 (log-likelihood per flight {'0.1': 55.20650122427764, '0.3': 68.85481320976898, '1.0': 89.5283227937693, '3.0': 93.27124140266112, '10.0': 93.26705249888624})
observability: median condition number of the whitened state Jacobian 19.6
    merged_with                                               states excited own_parameters_merged min_unique_share identifiable                           reason
fan       [lpt]              [fan_eff_mod+LPT_eff_mod, fan_flow_mod]    True                 False         0.409032        False       indistinguishable from lpt
lpc          []                          [LPC_eff_mod, LPC_flow_mod]    True                 Fal

## 9. Causal features (pipeline.md section 6) + leakage audit

In [39]:
stage("Features")
temporal_cols, static_cols = nc.default_feature_columns(len(ekf.state_names))
feat = nc.CausalFeaturizer(temporal_cols, static_cols, roll=ROLL).fit(cyc[train_m])    # medians from training only
frame = feat.transform(cyc)                                    # same row order as cyc (sorted by unit, cycle)
assert (frame["unit"].to_numpy() == groups).all()
FEATURES = nc.audit_features(feat.feature_names)
X = frame[FEATURES]
y = cyc["rul"].to_numpy(dtype=np.float64)
Xtr, ytr, Xva, yva, Xte, yte = X.iloc[tr_idx], y[tr_idx], X.iloc[va_idx], y[va_idx], X.iloc[te_idx], y[te_idx]
blocks = pd.Series([nc.feature_group(f) for f in FEATURES]).value_counts().to_dict()
print(f"{len(FEATURES)} features by block: {blocks}; window features: {sum(map(nc.is_window_feature, FEATURES))}")


=== Features   [7.8 min elapsed]
307 features by block: {'res': 84, 'phys': 72, 'raw': 70, 'ekf': 56, 'innov': 14, 'cond': 11}; window features: 170


## 10. Four base regressors (pipeline.md section 7): tuning on validation engines, out-of-fold predictions

In [40]:
stage("Base models")
t0 = time.time()
NAMES = list(st.BASE_NAMES)
PARAMS, TUNING = {}, []
for name in NAMES:
    grid = st.TUNE_GRIDS[name] if TUNE else [{}]               # XGBoost always early-stops on validation engines
    best, table = st.tune_model(name, Xtr, ytr, Xva, yva, grid)
    PARAMS[name] = best
    TUNING += table
    print(f"{name:13s} chosen {best}  (validation RMSE {min(r['val_rmse'] for r in table):.2f}) "
          f"[{(time.time() - t0) / 60:.1f} min]")
oof = st.oof_predictions(NAMES, PARAMS, Xtr, ytr, folds)
print("out-of-fold RMSE per model:", {n: round(st.rmse(oof[:, i], ytr), 2) for i, n in enumerate(NAMES)})
models = st.fit_base_models(NAMES, PARAMS, Xtr, ytr)
base_va, base_te = st.predict_bases(models, NAMES, Xva), st.predict_bases(models, NAMES, Xte)
TIMINGS["base_models_s"] = time.time() - t0


=== Base models   [7.8 min elapsed]
randomforest  chosen {'max_features': 0.5, 'min_samples_leaf': 5}  (validation RMSE 7.26) [13.6 min]
xgboost       chosen {'max_depth': 4, 'n_estimators': 117}  (validation RMSE 7.41) [14.8 min]
lightgbm      chosen {'num_leaves': 15, 'n_estimators': 111}  (validation RMSE 7.69) [15.1 min]
catboost      chosen {'depth': 5, 'iterations': 118}  (validation RMSE 7.49) [15.7 min]
out-of-fold RMSE per model: {'randomforest': 9.16, 'xgboost': 9.11, 'lightgbm': 9.26, 'catboost': 9.27}


## 11. Ridge stacking (pipeline.md section 8): scaler on OOF predictions, alpha on validation engines

In [41]:
stage("Ridge stack")
stack = st.fit_stack(oof, ytr, NAMES, base_va, yva)
pred_va = np.clip(stack.predict(base_va), 0, None)
pred_te = np.clip(stack.predict(base_te), 0, None)
print(f"alpha = {stack.alpha} (validation RMSE per alpha {stack.diagnostics['alpha_val_rmse']})")
print("coefficients (scaled base predictions):", dict(zip(NAMES, np.round(stack.coef, 3))), "intercept", round(stack.intercept, 2))
print("base-prediction correlation (OOF):")
print(pd.DataFrame(stack.diagnostics["base_prediction_correlation"], index=NAMES, columns=NAMES).round(3))
print(f"validation RMSE: stack {st.rmse(pred_va, yva):.2f}; best single "
      f"{min(st.rmse(base_va[:, i], yva) for i in range(len(NAMES))):.2f}")


=== Ridge stack   [37.2 min elapsed]
alpha = 0.01 (validation RMSE per alpha {'0.01': 7.047188974998865, '0.1': 7.048256121156113, '1.0': 7.058117014741545, '10.0': 7.112943716826591, '100.0': 7.195442983529002, '1000.0': 7.413088679440382})
coefficients (scaled base predictions): {'randomforest': np.float64(8.136), 'xgboost': np.float64(13.845), 'lightgbm': np.float64(-8.054), 'catboost': np.float64(8.435)} intercept 38.73
base-prediction correlation (OOF):
              randomforest  xgboost  lightgbm  catboost
randomforest         1.000    0.994     0.994     0.990
xgboost              0.994    1.000     0.998     0.992
lightgbm             0.994    0.998     1.000     0.992
catboost             0.990    0.992     0.992     1.000
validation RMSE: stack 7.05; best single 7.26


## 12. Evaluation (pipeline.md section 10) - test engines used once, here

In [42]:
stage("Evaluation")
metrics = {"test_engines": int(len(test_units)), "test_flights": int(len(te_idx)),
           "validation_engines": int(len(val_units)), "train_engines": int(len(train_units))}
metrics["rul_stack"] = st.report(pred_te, yte)
metrics["rul_stack_validation"] = st.report(pred_va, yva)
for i, n in enumerate(NAMES):
    metrics[f"rul_{n}"] = st.report(np.clip(base_te[:, i], 0, None), yte)
rows = [{"model": "Ridge stack", **{k: metrics["rul_stack"][k] for k in ("rmse", "mae", "r2", "nasa_score_mean")}}] + \
       [{"model": n, **{k: metrics[f"rul_{n}"][k] for k in ("rmse", "mae", "r2", "nasa_score_mean")}} for n in NAMES]
print("TEST engines - RUL in flight cycles:")
print(pd.DataFrame(rows).set_index("model").round(3))

# Empirical interval per predicted-RUL range (pipeline.md 9): residuals of the validation engines plus engine-held-out
# residuals of the whole stack on the training engines (more engines than validation alone; slightly pessimistic)
cv_tr = np.clip(st.cv_stack_predictions(oof, ytr, folds, NAMES, stack.alpha), 0, None)
INTERVAL = st.banded_interval(np.concatenate([pred_va, cv_tr]), np.concatenate([yva - pred_va, ytr - cv_tr]),
                              INTERVAL_LEVEL)
lo, hi = st.apply_interval(pred_te, INTERVAL)
inside = (yte >= lo) & (yte <= hi)
metrics["interval"] = {**INTERVAL, "method": "empirical: residual quantiles per predicted-RUL range (validation engines "
                                             "+ engine-held-out stack residuals of training engines)",
                       "coverage_test": float(inside.mean()), "mean_width_test": float(np.mean(hi - lo)),
                       "coverage_test_by_phase": {g: float(inside[st.rul_phase(yte) == g].mean())
                                                  for g in np.unique(st.rul_phase(yte))}}
print(f"empirical {INTERVAL_LEVEL:.0%} interval per predicted-RUL range: test coverage {inside.mean():.1%}, "
      f"mean width {np.mean(hi - lo):.1f} cycles")
print(pd.DataFrame(INTERVAL["bands"]).round(2).to_string(index=False))
print("coverage by true phase:", {k: f"{v:.0%}" for k, v in metrics["interval"]["coverage_test_by_phase"].items()})

te_ds = cyc["dataset"].to_numpy()[te_idx]
metrics["by_phase"] = st.error_by_group(pred_te, yte, st.rul_phase(yte))
metrics["by_subset"] = st.error_by_group(pred_te, yte, te_ds)
metrics["by_flight_class"] = st.error_by_group(pred_te, yte, cyc["fc"].to_numpy()[te_idx])
metrics["validation_by_engine"] = st.error_by_group(pred_va, yva, groups[va_idx])
metrics["validation_by_phase"] = st.error_by_group(pred_va, yva, st.rul_phase(yva))
per_engine = pd.DataFrame(st.error_by_group(pred_te, yte, groups[te_idx])).rename(columns={"group": "unit"})
per_engine["dataset"] = [subset_of[u] for u in per_engine["unit"]]
print("\nby degradation phase:")
print(pd.DataFrame(metrics["by_phase"]).round(2).to_string(index=False))
print("\nby subset:")
print(pd.DataFrame(metrics["by_subset"]).round(2).to_string(index=False))
metrics["health_mapping"] = HEALTH_MAP
TIMINGS["evaluation_s"] = time.time() - T_START


=== Evaluation   [37.2 min elapsed]
TEST engines - RUL in flight cycles:
               rmse    mae     r2  nasa_score_mean
model                                             
Ridge stack   6.974  5.168  0.911            0.804
randomforest  6.799  4.849  0.916            0.739
xgboost       7.191  5.370  0.906            0.816
lightgbm      7.202  5.363  0.905            0.824
catboost      7.203  5.493  0.905            0.821
empirical 90% interval per predicted-RUL range: test coverage 93.4%, mean width 27.5 cycles
 pred_above  pred_up_to    low  high    n source
        NaN        20.0  -4.71  7.85 1189   band
       20.0        60.0 -13.28 17.68 2391   band
       60.0         NaN -18.25 18.63  955   band
coverage by true phase: {np.str_('early (RUL>60)'): '97%', np.str_('end_of_life (RUL<=20)'): '93%', np.str_('mid (20<RUL<=60)'): '92%'}

by degradation phase:
                group    n  rmse  mae  bias
       early (RUL>60)  583  8.37 6.56 -3.22
end_of_life (RUL<=20)  819  4.16 2

## 13. Ablation E0-E7 (pipeline.md section 10.2) - same engines, folds, hyperparameters; only the features differ

In [43]:
stage("Ablation")
t0 = time.time()
ablation = pd.DataFrame()
if RUN_ABLATION:
    fgroup = np.array([nc.feature_group(f) for f in FEATURES])
    fwin = np.array([nc.is_window_feature(f) for f in FEATURES])

    def cols_of(blocks_, temporal):
        keep = np.isin(fgroup, list(blocks_) + ["cond"]) & (temporal | ~fwin)
        return [f for f, k in zip(FEATURES, keep) if k]

    EXPERIMENTS = {   # id: (purpose, feature blocks, temporal features?, model)
        "E0": ("raw-sensor Random Forest (baseline)", ("raw",), False, "rf"),
        "E1": ("raw-sensor four-model stack (ensemble benefit)", ("raw",), False, "stack"),
        "E2": ("healthy residual features + stack (condition compensation)", ("raw", "res"), False, "stack"),
        "E3": ("physics residual / EKF state features + RF", ("raw", "res", "phys", "ekf", "innov"), False, "rf"),
        "E4": ("physics + EKF + four-model stack (full pipeline)", ("raw", "res", "phys", "ekf", "innov"), True, "stack"),
        "E5": ("full pipeline without EKF features", ("raw", "res", "phys"), True, "stack"),
        "E6": ("full pipeline without physics residuals", ("raw", "ekf"), True, "stack"),
        "E7": ("full pipeline without temporal features", ("raw", "res", "phys", "ekf", "innov"), False, "stack"),
    }
    abl_rows = []
    for eid, (purpose, blocks_, temporal, kind) in EXPERIMENTS.items():
        cols = cols_of(blocks_, temporal)
        if eid == "E4":                                         # identical to the production pipeline: reuse it
            p_va, p_te = pred_va, pred_te
        elif kind == "rf":
            m = st.make_model("randomforest", PARAMS["randomforest"]).fit(Xtr[cols], ytr)
            p_va, p_te = np.clip(m.predict(Xva[cols]), 0, None), np.clip(m.predict(Xte[cols]), 0, None)
        else:
            o = st.oof_predictions(NAMES, PARAMS, Xtr[cols], ytr, folds)
            ms = st.fit_base_models(NAMES, PARAMS, Xtr[cols], ytr)
            bv, bt = st.predict_bases(ms, NAMES, Xva[cols]), st.predict_bases(ms, NAMES, Xte[cols])
            sk = st.fit_stack(o, ytr, NAMES, bv, yva)
            p_va, p_te = np.clip(sk.predict(bv), 0, None), np.clip(sk.predict(bt), 0, None)
        r = st.report(p_te, yte)
        abl_rows.append({"id": eid, "experiment": purpose, "features": len(cols), "test_rmse": r["rmse"],
                         "test_mae": r["mae"], "test_r2": r["r2"], "test_nasa_mean": r["nasa_score_mean"],
                         "test_rmse_end_of_life": st.rmse(p_te[yte <= 20], yte[yte <= 20]),
                         "validation_rmse": st.rmse(p_va, yva)})
        print(f"{eid} {purpose:58s} {len(cols):4d} features  test RMSE {r['rmse']:.2f}   "
              f"[{(time.time() - t0) / 60:.1f} min]")
    ablation = pd.DataFrame(abl_rows)
    metrics["ablation"] = ablation.to_dict(orient="records")
    print("\nAblation on TEST engines (RUL, cycles):")
    print(ablation.round(3).to_string(index=False))
    print("Claim only what this table shows: the EKF helps if E5 is worse than E4; physics residuals help if E6 is "
          "worse than E4; temporal features help if E7 is worse than E4.")
TIMINGS["ablation_s"] = time.time() - t0


=== Ablation   [37.2 min elapsed]
E0 raw-sensor Random Forest (baseline)                          81 features  test RMSE 10.42   [0.7 min]
E1 raw-sensor four-model stack (ensemble benefit)               81 features  test RMSE 10.21   [4.3 min]
E2 healthy residual features + stack (condition compensation)   95 features  test RMSE 7.42   [8.7 min]
E3 physics residual / EKF state features + RF                  137 features  test RMSE 7.07   [9.9 min]
E4 physics + EKF + four-model stack (full pipeline)            307 features  test RMSE 6.97   [9.9 min]
E5 full pipeline without EKF features                          237 features  test RMSE 7.34   [20.8 min]
E6 full pipeline without physics residuals                     137 features  test RMSE 7.21   [27.0 min]
E7 full pipeline without temporal features                     137 features  test RMSE 7.23   [33.3 min]

Ablation on TEST engines (RUL, cycles):
id                                                 experiment  features  test_rmse  tes

[link text](https://)## 14. Component-health forecasts (EXTENSION beyond `pipeline.md`): health h flights ahead
`pipeline.md` defines RUL as the only ML target; this section adds direct multi-horizon forecasts of every component's
health. For each component and horizon h one LightGBM model predicts the CHANGE in health over the next h flights from
today's causal features (+ today's EKF health estimates); forecast = EKF health now + predicted change, clipped to
[0, 1]. Future labels (the true health h flights later, from the hidden parameters T) are used as targets only.
Compared on the same engines against two baselines - "no change" (persistence) and a straight-line trend of the EKF
health - and a forecast is marked `useful` only if it beats BOTH on the validation engines' degraded flights.
Flights without h further flights (the last h of each engine) have no label and are excluded from training and scoring.

In [44]:
stage("Forecasting")
t0 = time.time()
FORECAST = {}
if RUN_FORECAST:
    kf_cols = [f"kf_hi_{c}" for c in nc.COMPONENTS]
    FC_FEATURES = FEATURES + kf_cols                              # same column order the backend rebuilds
    XF = X.copy()
    for col in kf_cols:
        XF[col] = cyc[col].to_numpy()
    cyc_cycle = cyc["cycle"].to_numpy()
    fc_models, fc_pred_te, fc_intervals, fc_useful, fc_metrics, fc_rows = {}, {}, {}, {}, {}, []
    for c in nc.COMPONENTS:
        cur, true_now = cyc[f"kf_hi_{c}"].to_numpy(), cyc[f"hi_{c}"].to_numpy()
        fc_intervals[c], fc_useful[c], fc_metrics[c] = {}, {}, {}
        for h in FORECAST_HORIZONS:
            fut = st.future_values(groups, cyc_cycle, true_now, h)
            trend = st.trend_projection(groups, cyc_cycle, cur, h, ROLL)
            has = ~np.isnan(fut)
            tr_r, va_r, te_r = (i[has[i]] for i in (tr_idx, va_idx, te_idx))
            if len(tr_r) < 50:
                print(f"  {c} h={h}: only {len(tr_r)} training flights have a label {h} flights ahead - skipped")
                continue
            model = st.make_model("lightgbm", PARAMS["lightgbm"]).fit(XF.iloc[tr_r], (fut - cur)[tr_r])

            def predict(rows):
                return np.clip(cur[rows] + model.predict(XF.iloc[rows]), 0.0, 1.0)

            met = {part: st.forecast_metrics(predict(r), fut[r], cur[r], trend[r], true_now[r], FORECAST_DEGRADED_BELOW)
                   for part, r in (("validation", va_r), ("test", te_r))}
            fc_metrics[c][str(h)] = met
            if len(va_r) >= 20:                                   # empirical interval from validation residuals
                lo_, hi_ = st.empirical_interval(fut[va_r] - predict(va_r), INTERVAL_LEVEL)
                fc_intervals[c][str(h)] = {"low": lo_, "high": hi_}
            vd = met["validation"]["degraded"]
            fc_useful[c][str(h)] = bool(vd["n"] >= RELIABLE_MIN_CYCLES and (vd["skill_vs_persistence"] or -1) > 0
                                        and (vd["skill_vs_trend"] or -1) > 0)
            fc_models[(c, h)] = model
            fc_pred_te[(c, h)] = predict(te_idx)                  # every test flight (also those without a future label)
            td = met["test"]["degraded"]
            fc_rows.append({"component": c, "h": h, "useful": fc_useful[c][str(h)], "test_degraded_n": td["n"],
                            "forecast_pts": None if td["forecast_rmse"] is None else 100 * td["forecast_rmse"],
                            "persistence_pts": None if td["persistence_rmse"] is None else 100 * td["persistence_rmse"],
                            "trend_pts": None if td["trend_rmse"] is None else 100 * td["trend_rmse"],
                            "skill_vs_persistence": td["skill_vs_persistence"], "skill_vs_trend": td["skill_vs_trend"],
                            "ekf_reliability": RELIABILITY[c]["status"]})
    FORECAST = {"models": fc_models, "pred_te": fc_pred_te, "intervals": fc_intervals, "useful": fc_useful,
                "metrics": fc_metrics, "features": FC_FEATURES}
    metrics["forecast"] = fc_metrics
    print("Forecast RMSE in health points (out of 100) on TEST engines, flights whose current true health < "
          f"{FORECAST_DEGRADED_BELOW}:")
    print(pd.DataFrame(fc_rows).round(3).to_string(index=False))
    print("`useful` was decided on VALIDATION engines (beats both baselines); skill > 0 means better than the baseline. "
          "Forecasts of components with ekf_reliability 'low' inherit the weakness of today's estimate.")
TIMINGS["forecast_s"] = time.time() - t0


=== Forecasting   [70.5 min elapsed]
Forecast RMSE in health points (out of 100) on TEST engines, flights whose current true health < 0.9:
component  h  useful  test_degraded_n  forecast_pts  persistence_pts  trend_pts  skill_vs_persistence  skill_vs_trend ekf_reliability
      fan 10   False              115        13.433           33.580     20.759                 0.600           0.353             low
      fan 25   False               35        22.425           61.149     45.200                 0.633           0.504             low
      fan 50   False                0           NaN              NaN        NaN                   NaN             NaN             low
      lpc 10   False              133        13.100           31.803     17.741                 0.588           0.262              ok
      lpc 25   False               25        16.728           80.681     54.471                 0.793           0.693              ok
      lpc 50   False                0           NaN     

## 15. Explainability (pipeline.md section 11): SHAP, Ridge contributions, physics/EKF diagnostics

In [45]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import shap

stage("Explainability")
t0 = time.time()
w_eff = stack.effective_weights()
total = np.zeros(len(FEATURES))
for i, name in enumerate(NAMES):
    n_rows = min(SHAP_ROWS if name != "randomforest" else min(SHAP_ROWS, 100), len(Xtr))
    sample = Xtr.sample(n_rows, random_state=SEED)
    sv = np.asarray(shap.TreeExplainer(models[name]).shap_values(sample)).reshape(n_rows, -1)[:, :len(FEATURES)]
    total += w_eff[i] * np.abs(sv).mean(axis=0)               # weighted by d RUL / d model prediction
shap_df = pd.DataFrame({"feature": FEATURES, "group": [nc.feature_group(f) for f in FEATURES],
                        "source": [nc.source_of(f) for f in FEATURES], "mean_abs_shap": np.abs(total)})
shap_df = shap_df.sort_values("mean_abs_shap", ascending=False)
shap_df.to_csv("shap_importance.csv", index=False)
by_block = shap_df.groupby("group")["mean_abs_shap"].sum().sort_values(ascending=False)
contrib_te = pd.DataFrame(stack.contributions(base_te), columns=NAMES).abs().mean().to_dict()
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
top = shap_df.head(15)[::-1]
axes[0].barh(top["feature"], top["mean_abs_shap"], color="#0284c7")
axes[0].set_title("RUL: top features (|SHAP| x Ridge weight)")
axes[1].bar(by_block.index, by_block.values, color="#0f766e")
axes[1].set_title("RUL: importance by feature block")
plt.tight_layout()
plt.savefig("shap_summary.png", dpi=130)
print("importance by block:", by_block.round(3).to_dict())
print("mean |Ridge contribution| per model on test (cycles):", {k: round(v, 2) for k, v in contrib_te.items()})
print("top features:", shap_df.head(10)[["feature", "mean_abs_shap"]].round(3).to_dict(orient="records"))
metrics["mean_abs_ridge_contribution_test"] = contrib_te
TIMINGS["shap_s"] = time.time() - t0


=== Explainability   [71.0 min elapsed]
importance by block: {'res': 12.229, 'cond': 11.817, 'ekf': 6.379, 'phys': 1.413, 'raw': 0.172, 'innov': 0.045}
mean |Ridge contribution| per model on test (cycles): {'randomforest': 7.04, 'xgboost': 12.1, 'lightgbm': 7.1, 'catboost': 7.59}
top features: [{'feature': 'cycle', 'mean_abs_shap': 6.088}, {'feature': 'exposure_h', 'mean_abs_shap': 5.603}, {'feature': 'res_T50', 'mean_abs_shap': 1.484}, {'feature': 'res_T50__rmax', 'mean_abs_shap': 1.395}, {'feature': 'ekf_0__rmax', 'mean_abs_shap': 1.058}, {'feature': 'res_T48__rs', 'mean_abs_shap': 1.011}, {'feature': 'res_T48__rmax', 'mean_abs_shap': 0.763}, {'feature': 'res_T48', 'mean_abs_shap': 0.732}, {'feature': 'ekf_0__rmin', 'mean_abs_shap': 0.67}, {'feature': 'ekf_0__rm', 'mean_abs_shap': 0.643}]


## 16. Cost: timings and inference latency

In [46]:
one_unit = test_units[0]
cols_raw = [c for c in cyc.columns if not c.startswith(("ref_", "res_", "ekf_", "kf_", "innov_", "nis_", "gated_", "hi_"))]
t0 = time.time()
for _ in range(3):
    one = nc.run_layers(cyc.loc[cyc.unit == one_unit, cols_raw], ref, ekf, D_REF)
    stack.predict(st.predict_bases(models, NAMES, feat.transform(one)[FEATURES]))
metrics["inference_ms_one_engine_all_flights"] = (time.time() - t0) / 3 * 1000
TIMINGS["total_s"] = time.time() - T_START
metrics["timings_s"] = {k: round(v, 1) for k, v in TIMINGS.items()}
print(json.dumps(metrics["timings_s"], indent=1),
      f"\none engine, all flights, full pipeline: {metrics['inference_ms_one_engine_all_flights']:.0f} ms")

{
 "download_and_aggregate_s": 441.5,
 "layer2_s": 14.5,
 "layer3_s": 9.2,
 "base_models_s": 1764.5,
 "evaluation_s": 2231.6,
 "ablation_s": 1998.5,
 "forecast_s": 30.0,
 "shap_s": 11.0,
 "total_s": 4272.6
} 
one engine, all flights, full pipeline: 467 ms


## 17. Export the versioned bundle (pipeline.md 13.1), save a copy to Drive, download

In [47]:
import joblib

stage("Export")


def jclean(o):
    """JSON-safe copy: numpy scalars -> python, NaN/inf -> None."""
    if isinstance(o, dict):
        return {str(k): jclean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [jclean(v) for v in o]
    if isinstance(o, np.ndarray):
        return jclean(o.tolist())
    if isinstance(o, np.generic):
        o = o.item()
    if isinstance(o, float) and not np.isfinite(o):
        return None
    return o


def dump(obj, name):
    with open(os.path.join(OUT, name), "w") as fh:
        json.dump(jclean(obj), fh, indent=1)


OUT = f"bundle_{RUN}"
shutil.rmtree(OUT, ignore_errors=True)
os.makedirs(OUT)
EXT = {"randomforest": "joblib", "xgboost": "json", "lightgbm": "txt", "catboost": "cbm"}
base_entries = []
for name in NAMES:
    fn = f"{name}_rul.{EXT[name]}"
    m = models[name]
    if name == "randomforest":
        joblib.dump(m, os.path.join(OUT, fn), compress=3)
    elif name == "lightgbm":
        m.booster_.save_model(os.path.join(OUT, fn))
    else:
        m.save_model(os.path.join(OUT, fn))
    base_entries.append({"name": name, "file": fn, "params": {**st.SPEC_PARAMS[name], **PARAMS[name]}})
dump(ref.to_dict(), "reference.json")
dump(ekf.to_dict(), "ekf.json")
dump(feat.to_dict(), "featurizer.json")
dump(stack.to_dict(), "stack.json")
FORECAST_MANIFEST = None
if FORECAST:
    fc_files = {}
    for (c_, h_), m_ in FORECAST["models"].items():
        fn_ = f"forecast_{c_}_h{h_}.txt"
        m_.booster_.save_model(os.path.join(OUT, fn_))
        fc_files.setdefault(c_, {})[str(h_)] = fn_
    FORECAST_MANIFEST = {"model": "lightgbm", "horizons": list(FORECAST_HORIZONS), "features": FORECAST["features"],
                         "target": "true health h flights ahead minus the current EKF health estimate",
                         "files": fc_files, "intervals": FORECAST["intervals"], "useful": FORECAST["useful"],
                         "degraded_below": FORECAST_DEGRADED_BELOW, "metrics": FORECAST["metrics"],
                         "note": "EXTENSION beyond pipeline.md; use a forecast only where useful is true"}
dump({"sensors": {s: {"description": nc.SENSOR_INFO[s][0], "unit": nc.UNITS[nc.SENSOR_INFO[s][1]]} for s in nc.XS_VARS},
      "conditions": {w: nc.UNITS[w] for w in nc.W_VARS}, "hidden_parameters_offline_only": nc.T_VARS,
      "inference_cadence": "flight_end", "features": FEATURES, "feature_blocks": blocks, "files": SCHEMA}, "schema.json")
shutil.copy("shap_importance.csv", OUT)
shutil.copy("shap_summary.png", OUT)
per_engine.to_csv(os.path.join(OUT, "per_engine_metrics.csv"), index=False)
pd.DataFrame(metrics["by_subset"]).to_csv(os.path.join(OUT, "per_subset_metrics.csv"), index=False)
res_band.to_csv(os.path.join(OUT, "healthy_residuals_by_altitude.csv"), index=False)
if len(ablation):
    ablation.to_csv(os.path.join(OUT, "ablation.csv"), index=False)

# Demo data for the dashboard = the held-out TEST engines (per-flight aggregates; the backend recomputes everything)
demo_cols = ["unit", "unit_local", "dataset", "cycle", "fc", "hs", "rul", "duration_s", "alt_max", "Mach_max"] + \
    [f"{w}_mean" for w in nc.W_VARS] + [c for c in cyc.columns if c.startswith(nc.SUMMARY_PREFIXES + ("p_", "v_", "phi_"))] + \
    nc.T_VARS + [f"hi_{c}" for c in nc.COMPONENTS]
cyc.loc[test_m, list(dict.fromkeys(c for c in demo_cols if c in cyc.columns))].to_csv(
    os.path.join(OUT, "demo_cycles.csv"), index=False)
par = cyc.loc[test_m, ["unit", "cycle"]].copy()
par["pred_rul"] = pred_te
for i, n in enumerate(NAMES):
    par[f"pred_base_{n}"] = base_te[:, i]
for c in nc.COMPONENTS:
    par[f"kf_hi_{c}"] = cyc.loc[test_m, f"kf_hi_{c}"].to_numpy()
for (c, h), v_ in (FORECAST.get("pred_te") or {}).items():
    par[f"pred_fc_{c}_h{h}"] = v_
par.to_csv(os.path.join(OUT, "parity_outputs.csv"), index=False)

import catboost
import lightgbm
import sklearn
import xgboost
versions = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
            "scikit-learn": sklearn.__version__, "xgboost": xgboost.__version__, "lightgbm": lightgbm.__version__,
            "catboost": catboost.__version__, "shap": shap.__version__, "joblib": joblib.__version__}
dump(versions, "versions.json")
config = {k: globals()[k] for k in ("RUN", "SEED", "VAL_FRACTION", "N_FOLDS", "ROLL", "MERGE_COS", "MERGE_UNIQUE", "EKF_GATE", "Q_GRID",
                                    "TUNE", "INTERVAL_LEVEL", "RELIABLE_MAX_RMSE", "RELIABLE_MIN_CYCLES",
                                    "HEALTHY_SAMPLES_PER_FILE")}
manifest = {
    "version": 4, "spec": "pipeline.md", "dataset": "NASA N-CMAPSS", "run": RUN, "files": FILES,
    "components": nc.COMPONENTS, "targets": ["rul"], "d_ref": D_REF, "band_edges": BAND_EDGES, "features": FEATURES,
    "reference_file": "reference.json", "ekf_file": "ekf.json", "featurizer_file": "featurizer.json",
    "stack_file": "stack.json", "schema_file": "schema.json", "base_models": base_entries,
    "interval": metrics["interval"], **({"forecast": FORECAST_MANIFEST} if FORECAST_MANIFEST else {}), "demo_file": "demo_cycles.csv", "parity_file": "parity_outputs.csv",
    "splits": {"train_engines": train_units, "validation_engines": val_units, "test_engines": test_units},
    "dataset_fingerprints": FINGERPRINTS, "observability": OBS, "ekf_diagnostics": EKF_DIAG,
    "health_mapping": HEALTH_MAP, "reliability": RELIABILITY, "tuning": TUNING, "metrics": metrics, "config": config,
    "versions": versions, "load_log": load_log,
    "build": {**BUILD_INFO, "python": platform.python_version(),
              "trained_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())},
    "notes": ["RUL = Ridge stack (scaler fitted on out-of-fold predictions) of RF, XGBoost, LightGBM, CatBoost.",
              "Component health = EKF parameter estimates mapped as 1 - damage / d_ref; not an ML target. Show only "
              "components whose 'reliability' status is ok as component health; others as low-confidence.",
              "RUL interval is empirical (validation residual quantiles), not a calibrated probability.",
              "Hidden parameters T were used offline only (EKF calibration, health-mapping validation).",
              "Research prototype - not for maintenance decisions."],
}
dump(manifest, "manifest.json")
dump(metrics, "metrics.json")
zip_path = shutil.make_archive(f"genzone_ncmapss_{RUN}_bundle", "zip", OUT)
print("Bundle:", zip_path, f"({os.path.getsize(zip_path) / 1e6:.1f} MB); total time {(time.time() - T_START) / 60:.1f} min")
if CACHE_DIR:
    shutil.copy(zip_path, CACHE_DIR)
    print("copy saved to", CACHE_DIR)
print("\nNEXT: unzip into backend/models/, run `python -m backend.check_parity` (must print PARITY OK), and copy the "
      "tables of sections 8, 11, 12, 13 and 14 into the report.")
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)


=== Export   [71.2 min elapsed]
Bundle: /content/genzone_ncmapss_full_bundle.zip (11.6 MB); total time 71.3 min
copy saved to /content/drive/MyDrive/genzone

NEXT: unzip into backend/models/, run `python -m backend.check_parity` (must print PARITY OK), and copy the tables of sections 8, 11, 12, 13 and 14 into the report.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>